# Memoria, contexto y grafos — Notebook A: memoria (Clase 9)

*Para el modelo, lo que no está en la ventana no existe.*

Este notebook acompaña al deck de la Clase 9, que ocupa dos semanas: las secciones 1 a 8 van con la semana 9, y de la 9 en adelante, con la semana 10. El deck **enuncia**; acá **exhibimos** cada mecanismo una vez, corriendo de arriba a abajo.

**Claude se acuerda; el modelo, no.** Claude sigue el hilo de una conversación, resume lo viejo cuando un chat se alarga y, en un chat nuevo, se acuerda de lo que se le contó otro día. El modelo no hace nada de eso: no guarda nada de una llamada a la otra. Lo hace el programa que lo rodea, el ***harness***, que decide qué texto le llega al modelo en cada llamada. Con la API, el harness lo escribimos nosotros: **este notebook es ese harness, armado de a una pieza por vez**.

La clase tiene un solo hilo: todo lo que un producto llama memoria es el harness decidiendo **qué vuelve a entrar a la ventana** en cada llamada. Y la pregunta que corre sobre ese hilo tiene tres partes: qué vuelve a entrar, quién lo decide y **cuándo** —al escribir, sin conocer la pregunta que va a venir, o al leer, cuando ya se la conoce—.

**El recorrido** sigue una sola conversación, la de un usuario que trabaja en una pesquera de Vigo. A medida que crece se rompe algo distinto, y cada arreglo deja el problema siguiente: son los seis momentos del deck, los que dice el pie de cada lámina. Las secciones llevan el número del deck y el nombre de su tramo.

**Antes de empezar**

0. Preparación: el entorno, el generador y el pipeline de la clase de RAG, en versión compacta.

**Momento 1 · La siguiente pregunta**

1. El modelo no recuerda: sin estado, la plantilla de chat y las dos memorias del modelo.
2. Buscar en una conversación: con el turno solo, con la conversación entera o con la pregunta reescrita.

**Momento 2 · La conversación larga**

3. Mandar todo cuesta: el turno 20, el prefill y el decode, el KV cache, el cache de prefijos.
4. La ventana se acaba: el desborde silencioso, provocado a propósito.
5. Repartir la ventana: el ensamblador y su presupuesto.

**Momento 3 · Cuando ya no entra**

6. Qué sacar de la ventana: recortar, anclar, resumir y descargar, sobre un hecho plantado.
7. Resumir por partes: la pregunta global, y las cadenas de resumen.
8. Lo que el resumen pierde: el identificador perdido.

**Momento 4 · Al día siguiente** (semana 10)

9. Al día siguiente: tres sesiones, y las dos vallas.
10. Qué se escribe: extraer hechos con un schema.
11. Mantener la memoria al día: consolidar, e invalidar en vez de borrar.
12. Leer la memoria: relevancia, recencia e importancia, y la trampa de normalizar.
13. Medir la memoria: una vara por habilidad.
14. Dónde falla la memoria: un registro falso que se lee para siempre.

**Momento 5 · Datos conectados.** Las secciones 15 a 19 —la memoria con estructura: Graph RAG— están en el notebook B, `Clase09_nbB_grafos.ipynb`.

**Momento 6 · ¿Quién decide?**

20. Quién decide qué recordar: la memoria como herramienta.

**Cierre**

21. El mismo sistema en un framework, y el cierre.

**Cómo está armado.** Todo libre y local. El corpus es el de la clase de RAG —actos del derecho europeo en español, de MultiEURLEX—, el embedder es `BAAI/bge-m3`, el canal léxico es `bm25s`, el generador es `qwen3:8b` corriendo en Ollama, y la sección 21 usa LangGraph. Sin APIs pagas y sin claves.

**Cuánto tarda.** Con `SMOKE_TEST = True` son unos 25 minutos más las descargas; con `False`, unos 45. Casi todo el notebook habla con el generador: si el bootstrap de Ollama falla, las celdas que lo necesitan lo dicen y siguen de largo, pero no queda mucho para mirar.

> **Activá la GPU:** *Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU.*

## 0 · Preparación

Primero miramos qué hardware tocó. La respuesta cambia los tiempos, no lo que se puede correr: casi todo funciona en CPU, sólo que mucho más lento.

In [1]:
# Que tenemos. Si no lista ninguna GPU, activala en Entorno de ejecucion -> T4 GPU.
!nvidia-smi --query-gpu=name,memory.total,memory.used --format=csv,noheader || echo "sin GPU"

Tesla T4, 15360 MiB, 0 MiB


**Todas las instalaciones van juntas y acá arriba.** Un `pip install` que actualiza algo que Colab trae precargado puede pedir **reiniciar el entorno**; si la instalación estuviera repartida por el notebook, ese reinicio llegaría con el índice ya construido y se perdería todo. Las versiones van con número exacto, por la misma razón que en la clase de RAG: una versión de más puede romper un notebook entero.

In [2]:
# Un solo install, con numeros exactos. Si alguna linea falla, el resto igual se instala.
!pip install -q \
  "sentence-transformers==5.7.0" \
  "bm25s==0.3.11" "PyStemmer==3.1.0" \
  "langchain==1.4.3" "langgraph==1.2.14" "langchain-ollama==1.1.0" \
  "huggingface_hub>=0.23" "pyarrow>=21"

# Por que estas y no otras:
#  - sentence-transformers 5.7.0 es la de la clase de RAG: convive con el transformers que
#    trae Colab sin forzar un upgrade grande.
#  - bm25s y PyStemmer arman el canal lexico, igual que en RAG.
#  - langchain, langgraph y langchain-ollama solo los usa la seccion 21. Van aca arriba igual,
#    porque son los que mas probablemente pidan reiniciar el entorno.
#  - El cliente de Ollama no lo instalamos aparte: le hablamos a su API con requests, que
#    ya viene en Colab, para poder pasarle campos que el cliente no expone (seccion 4).

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 747.7/747.7 kB 19.9 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 256.0/256.0 kB 27.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.4/162.4 kB 18.1 MB/s eta 0:00:00


> **Si Colab pide reiniciar el entorno, reiniciá y seguí desde acá.** No hace falta volver a correr la celda de instalación.

In [3]:
# Las versiones que realmente quedaron. Casi todos los errores raros de este stack son una
# version distinta de la que dice arriba.
import importlib.metadata as meta
for p in ("sentence-transformers", "transformers", "bm25s", "langchain", "langgraph",
          "langchain-ollama", "langchain-core", "huggingface_hub", "numpy"):
    try:
        print("%-24s %s" % (p, meta.version(p)))
    except Exception:
        print("%-24s (no instalado)" % p)

sentence-transformers    5.7.0
transformers             5.18.0
bm25s                    0.3.11
langchain                1.4.3
langgraph                1.2.14
langchain-ollama         1.1.0
langchain-core           1.6.6
huggingface_hub          1.33.0
numpy                    2.1.3


### 0.1 · Ollama

El generador del curso corre local: instalar, levantar el servidor en segundo plano y bajar el modelo. Dejamos una bandera, `OLLAMA_OK`: si el bootstrap falla, el notebook no se cae; las celdas que necesitan el generador lo dicen y siguen.

Dos decisiones de esta celda que la clase usa después. **Un solo pedido a la vez** (`OLLAMA_NUM_PARALLEL=1`): con más, el servidor reparte la ventana y el cache entre pedidos simultáneos, y la sección 3 mide un único cache. Y **no fijamos la ventana del servidor**, a propósito: la sección 4 necesita ver qué ventana asigna Ollama cuando nadie se la pide.

In [4]:
# zstd hace falta para descomprimir el instalador de Ollama en Colab.
!apt-get -qq update && apt-get -qq install -y zstd
!curl -fsSL https://ollama.com/install.sh | sh

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu noble InRelease' does not seem to provide it (sources.list entry misspelt?)
Selecting previously unselected package zstd.
(Reading database ... 127332 files and directories currently installed.)
Preparing to unpack .../zstd_1.5.5+dfsg2-2build1.1_amd64.deb ...
Unpacking zstd (1.5.5+dfsg2-2build1.1) ...
Setting up zstd (1.5.5+dfsg2-2build1.1) ...
Processing triggers for man-db (2.12.0-4build2) ...
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


In [5]:
import os, subprocess, time, requests

os.environ["OLLAMA_HOST"] = "127.0.0.1:11434"
os.environ["OLLAMA_MAX_LOADED_MODELS"] = "1"   # un solo modelo en la placa: el generador
os.environ["OLLAMA_NUM_PARALLEL"] = "1"        # un pedido a la vez: un solo cache que medir
# OLLAMA_CONTEXT_LENGTH no se fija: la seccion 4 mira el default del servidor.

# JAX entra de prestado con bm25s y, si inicializa su backend en la GPU, PREASIGNA EL 75%
# DE LA VRAM. Ni PyTorch ni Ollama lo ven: el generador encuentra la placa llena y se
# reparte con el procesador, sin ningun error. Tiene que estar antes de importar bm25s.
os.environ["XLA_PYTHON_CLIENT_PREALLOCATE"] = "false"
os.environ["JAX_PLATFORMS"] = "cpu"

URL_OLLAMA = "http://127.0.0.1:11434"
LOG_OLLAMA = "ollama.log"     # lo que dice el servidor: si algo se cuelga, el porque esta aca

def ollama_vivo():
    try:
        return requests.get(URL_OLLAMA + "/api/tags", timeout=2).ok
    except requests.exceptions.RequestException:
        return False

if ollama_vivo():
    print("Ollama ya estaba corriendo.")
else:
    # Barrido previo: un runner huerfano de una corrida anterior puede seguir reservando
    # VRAM, y el servidor nuevo carga el modelo en CPU sin avisar.
    subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
    time.sleep(3)
    # start_new_session=True: sin eso, interrumpir una celda le manda el SIGINT tambien al
    # servidor, y el error aparece recien varias secciones despues.
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open(LOG_OLLAMA, "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            print("Ollama está listo.")
            break
        time.sleep(1)
    else:
        print("El servidor tardó en levantar; volvé a correr esta celda.")

OLLAMA_OK = ollama_vivo()
print("OLLAMA_OK =", OLLAMA_OK)

Ollama está listo.
OLLAMA_OK = True


In [6]:
MODEL = "qwen3:8b"     # el generador del curso. El tag va siempre explicito: sin tag es "latest"

if OLLAMA_OK:
    print("pull", MODEL, "...", flush=True)
    r = subprocess.run(["ollama", "pull", MODEL], capture_output=True, text=True)
    if r.returncode != 0:
        print("   FALLO:", (r.stderr or r.stdout).strip()[:200])
        OLLAMA_OK = False
    else:
        !ollama list
        print(subprocess.run(["ollama", "--version"], capture_output=True, text=True).stdout.strip())
else:
    print("Sin Ollama: las celdas que generan van a marcarlo y seguir de largo.")

pull qwen3:8b ...
NAME        ID              SIZE      MODIFIED               
qwen3:8b    500a1f067a9f    5.2 GB    Less than a second ago    
ollama version is 0.40.3


### 0.2 · Banderas y ayudantes

`SMOKE_TEST` es la convención del curso. Acá achica **sólo el corpus** sobre el que busca el pipeline de RAG —514 actos en vez de 1.544, el mismo prefijo que en la clase de RAG—: indexar es lo único caro que no es el generador. Todo lo demás corre igual. Los números de las láminas salen de la corrida con `False`.

In [7]:
import re, json, random, datetime
import numpy as np

SMOKE_TEST = False     # True = 514 actos (~25 min). False = 1.544, la corrida de las laminas (~45 min).

N_ACTOS      = 514 if SMOKE_TEST else 1544
TAM_PEDAZO   = 350     # tokens por pedazo, medidos con el tokenizer del embedder
SOLAPE       = 50
K            = 5       # pasajes que le llegan al generador
N_CANDIDATOS = 50      # cuantos trae cada canal antes de fusionar
NUM_CTX      = 8192    # la ventana que pedimos en cada llamada, salvo donde se dice otra cosa

random.seed(1337)
np.random.seed(1337)

RESULTADOS = {}        # lo que se mide, por seccion. El cierre lo imprime desde aca.
FALLAS_GENERADOR = []  # llamadas que fallaron; el cierre las reporta

print("modo:", "SMOKE_TEST (barato)" if SMOKE_TEST else "COMPLETO", "·", N_ACTOS, "actos")

modo: COMPLETO · 1544 actos


**Una sola función para hablar con el generador.** `chat()` manda una lista de mensajes a la API de chat de Ollama y devuelve, además de la respuesta, **los contadores del servidor**: cuántos tokens leyó del prompt, cuánto tardó en leerlos (el *prefill*), cuántos escribió y cuánto tardó en escribirlos (el *decode*). Con esos cuatro números se mide el costo de la conversación larga, el momento 2 de la clase.

Tres decisiones, cortas. Temperatura cero, para que la corrida se pueda repetir. `think=False`: Qwen3 es un modelo híbrido, y su razonamiento no es el tema de hoy. Y **nunca una excepción**: si una llamada falla, queda una marca en la salida y el notebook sigue. Un rechazo del servidor tampoco es una excepción: es un dato, y la sección 4 lo usa.

In [8]:
TIMEOUT_GEN = 300    # segundos por llamada. La mas larga del notebook tarda menos de un minuto.

def asegurar_ollama(forzar=False):
    """Si el servidor se murio en el medio, lo levanta otra vez. Con forzar=True lo reinicia
    aunque conteste: un servidor colgado sigue respondiendo /api/tags y no genera nada."""
    if not OLLAMA_OK or (ollama_vivo() and not forzar):
        return True
    if forzar:
        try:
            ultimas = open(LOG_OLLAMA, errors="replace").read().splitlines()[-3:]
            print("Ollama no termina de generar. Lo último de su log:")
            for l in ultimas:
                print("   " + l[:150])
        except OSError:
            pass
        subprocess.run(["pkill", "-9", "-f", "ollama"], check=False)
        time.sleep(3)
    print("Lo vuelvo a levantar...")
    subprocess.Popen(["ollama", "serve"], start_new_session=True,
                     stdout=open(LOG_OLLAMA, "a"), stderr=subprocess.STDOUT)
    for _ in range(60):
        if ollama_vivo():
            return True
        time.sleep(1)
    print("No pude levantarlo.")
    return False

def chat(mensajes, fmt=None, num_ctx=NUM_CTX, num_predict=600, temperature=0.0,
         herramientas=None, extra=None):
    """Una llamada a /api/chat. Devuelve la respuesta Y los contadores del servidor.

    Recibe:
      mensajes      la lista [{role, content}, ...], tal cual se manda
      fmt           un JSON schema para structured outputs, o None
      num_ctx       la ventana que pedimos. None NO la pide, y el servidor usa la suya
      num_predict   tope de tokens de la respuesta
      herramientas  tools para function calling (seccion 20)
      extra         campos sueltos del pedido, como truncate o shift (seccion 4)

    Devuelve {texto, mensaje, prompt, prefill_s, salida, decode_s, error}:
      prompt     tokens del prompt SEGUN EL SERVIDOR, incluidos los que vinieron del cache
      prefill_s  segundos leyendo el prompt: solo lo que NO estaba en cache
      salida     tokens generados, y decode_s los segundos que tardo en generarlos
      error      None, o el texto del rechazo si el servidor no acepto el pedido
    """
    vacio = {"texto": "", "mensaje": {}, "prompt": 0, "prefill_s": 0.0,
             "salida": 0, "decode_s": 0.0, "error": None}
    if not OLLAMA_OK:
        return dict(vacio, texto="[sin Ollama: esta celda no corrió]", error="sin Ollama")
    opciones = {"temperature": temperature, "num_predict": num_predict}
    if num_ctx:
        opciones["num_ctx"] = num_ctx
    cuerpo = {"model": MODEL, "messages": mensajes, "stream": False, "think": False,
              "options": opciones}
    if fmt:
        cuerpo["format"] = fmt
    if herramientas:
        cuerpo["tools"] = herramientas
    if extra:
        cuerpo.update(extra)
    for intento in (1, 2):
        try:
            r = requests.post(URL_OLLAMA + "/api/chat", json=cuerpo, timeout=TIMEOUT_GEN)
        except requests.exceptions.RequestException as e:
            FALLAS_GENERADOR.append("%s: %s" % (type(e).__name__, str(e)[:80]))
            if intento == 1:
                # Un timeout es un servidor colgado: se reinicia aunque conteste.
                asegurar_ollama(forzar=isinstance(e, requests.exceptions.Timeout))
                continue
            print("   ⚠ el generador no respondió (%s). Sigo." % type(e).__name__)
            return dict(vacio, texto="[el generador no respondió]", error=type(e).__name__)
        if r.status_code != 200:
            return dict(vacio, error="HTTP %d · %s" % (r.status_code, r.text[:400]))
        j = r.json()
        msg = j.get("message") or {}
        texto = msg.get("content") or ""
        # Qwen3 a veces filtra el cierre de su razonamiento aun con think=False.
        if "</think>" in texto:
            texto = texto.split("</think>")[-1]
        return {"texto": texto.strip(), "mensaje": msg,
                "prompt": j.get("prompt_eval_count", 0),
                "prefill_s": j.get("prompt_eval_duration", 0) / 1e9,
                "salida": j.get("eval_count", 0),
                "decode_s": j.get("eval_duration", 0) / 1e9, "error": None}
    return dict(vacio, texto="[el generador no respondió]", error="reintentos")

def ask(prompt, system=None, **kw):
    """Una pregunta suelta, sin historial. Devuelve solo el texto."""
    msgs = ([{"role": "system", "content": system}] if system else []) + \
           [{"role": "user", "content": prompt}]
    return chat(msgs, **kw)["texto"]

def estructurado(mensajes, esquema, num_predict=600, **kw):
    """Structured output con un modelo de pydantic. Devuelve (objeto o None, llamada cruda)."""
    r = chat(mensajes, fmt=esquema.model_json_schema(), num_predict=num_predict, **kw)
    try:
        return esquema.model_validate_json(r["texto"]), r
    except Exception:
        return None, r

if OLLAMA_OK:
    print(ask("Respondé en una sola palabra: ¿capital de Francia?"))

París


**Cuatro ayudantes de impresión**, los mismos de la clase de RAG: `tabla()` para los resultados, `mostrar()` para un bloque con título, `parrafo()` para leer un texto largo sin scrollear de lado, y `recortar()` para que cada ítem de una lista ocupe un renglón.

In [9]:
import textwrap

def tabla(titulo, columnas, filas):
    """Imprime una tabla chiquita. filas = [(nombre, {columna: valor}), ...]."""
    w0 = max([len(str(f[0])) for f in filas] + [10]) + 2
    anchos = [max([len(str(f[1].get(c, "—"))) for f in filas] + [len(str(c))]) + 3
              for c in columnas]
    print("\n" + titulo)
    print("─" * (w0 + sum(anchos)))
    print("".ljust(w0) + "".join(str(c).rjust(w) for c, w in zip(columnas, anchos)))
    for nombre, vals in filas:
        print(str(nombre).ljust(w0) +
              "".join(str(vals.get(c, "—")).rjust(w) for c, w in zip(columnas, anchos)))
    print()

def parrafo(texto, ancho=74, sangria="   ", lineas=None):
    """Un texto largo, envuelto en renglones cortos. Respeta los saltos que ya traiga."""
    renglones = []
    for original in re.sub(r"[ \t]+", " ", str(texto).strip()).split("\n"):
        renglones += textwrap.wrap(original, width=ancho) or [""]
    if lineas and len(renglones) > lineas:
        renglones = renglones[:lineas]
        renglones[-1] += " …"
    return "\n".join(sangria + r for r in renglones)

def mostrar(titulo, cuerpo):
    """Un bloque con titulo, para los antes-y-despues."""
    print("─" * 74)
    print(titulo)
    print("─" * 74)
    print(parrafo(cuerpo, sangria=""))
    print()

def recortar(texto, n=160):
    """Una linea, sin saltos, para meter un texto adentro de una lista."""
    limpio = re.sub(r"\s+", " ", str(texto)).strip()
    return limpio[:n] + ("…" if len(limpio) > n else "")

**Y los ayudantes de la placa.** En una T4 conviven dos runtimes que no se ven entre sí: PyTorch, que carga el embedder, y Ollama, que carga el generador. `vram()` dice quién tiene la memoria, `liberar_gpu()` devuelve la que PyTorch retiene, y `donde_corre()` confirma que el generador entró en la GPU y no en el procesador. `modelo_en_placa()` le pregunta al servidor cuánto ocupa el modelo cargado y con qué ventana: la sección 3 lo usa para medir el KV cache.

In [10]:
import torch

DISPOSITIVO = "cuda" if torch.cuda.is_available() else "cpu"

def vram(etiqueta=""):
    """Libre en la placa, reservado por PyTorch y atribuido a este proceso."""
    if not torch.cuda.is_available():
        return
    libre = subprocess.run(["nvidia-smi", "--query-gpu=memory.free",
                            "--format=csv,noheader,nounits"],
                           capture_output=True, text=True).stdout.strip()
    print("   [vram] %-26s libre %6s MiB · PyTorch %5.1f GB"
          % (etiqueta, libre, torch.cuda.memory_reserved() / 1e9))

def liberar_gpu(modelo=None):
    """Devuelve la VRAM que PyTorch retiene y, si se pide, descarga el modelo de Ollama."""
    import gc
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    if modelo and OLLAMA_OK:
        subprocess.run(["ollama", "stop", modelo], check=False, capture_output=True)

def donde_corre():
    """La columna que importa es PROCESSOR: '100% GPU' es lo esperado."""
    r = subprocess.run(["ollama", "ps"], capture_output=True, text=True)
    print(r.stdout.rstrip() if r.returncode == 0 else "OLLAMA NO RESPONDE")

def modelo_en_placa():
    """Lo que el servidor dice del modelo cargado: bytes totales, bytes en GPU y ventana."""
    try:
        for m in requests.get(URL_OLLAMA + "/api/ps", timeout=5).json().get("models", []):
            if m.get("name", "").startswith(MODEL) or m.get("model", "").startswith(MODEL):
                return {"total": m.get("size", 0), "gpu": m.get("size_vram", 0),
                        "ventana": m.get("context_length")}
    except Exception:
        pass
    return None

print("ayudantes de placa listos · dispositivo:", DISPOSITIVO)

ayudantes de placa listos · dispositivo: cuda


### 0.3 · Contar como cuenta el servidor

El tamaño se mide en tokens, y en los tokens **del modelo que lee**. Ollama no expone su tokenizer, así que bajamos el de Qwen3 de HuggingFace: es el mismo vocabulario y la misma plantilla de chat. Con él, `contar_prompt()` reproduce el prompt exacto que arma el servidor —incluido un detalle que Ollama agrega por su cuenta: con `think=False`, le pega `/no_think` al último mensaje del usuario—. La sección 4 compara esta cuenta con la del servidor, y la diferencia es la que delata un recorte.

In [11]:
from transformers import AutoTokenizer
import transformers
transformers.logging.set_verbosity_error()

TOK_GEN = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")   # solo el tokenizer: ~11 MB

def contar(texto):
    """Tokens de un texto suelto, con el tokenizer del generador."""
    return len(TOK_GEN.encode(texto, add_special_tokens=False))

def plantilla(mensajes, generar=True):
    """La lista de mensajes convertida en el UNICO texto que el modelo lee."""
    return TOK_GEN.apply_chat_template(mensajes, tokenize=False,
                                       add_generation_prompt=generar,
                                       enable_thinking=False)

def contar_prompt(mensajes):
    """Los tokens del prompt como los cuenta el servidor: misma plantilla, mismo vocabulario,
    y el ' /no_think' que Ollama agrega al ultimo turno del usuario cuando think=False."""
    msgs = [dict(m) for m in mensajes]
    for m in reversed(msgs):
        if m["role"] == "user":
            m["content"] = m["content"] + " /no_think"
            break
    return contar(plantilla(msgs))

print("«¿Y qué dice sobre el bacalao?» son", contar("¿Y qué dice sobre el bacalao?"), "tokens del generador")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/9.73k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

tokenizer.json: downloading bytes:           |  0.00B            

«¿Y qué dice sobre el bacalao?» son 10 tokens del generador


### 0.4 · El pipeline de RAG, compacto

La conversación de hoy corre sobre el pipeline de la clase de RAG: el mismo corpus, el mismo corte por estructura con el título del acto adelante de cada pedazo, el mismo embedder y el mismo canal léxico, fusionados por puesto. Quedan afuera el re-puntuador y la subida al padre: le dejan la placa al generador, y hoy el tema no es recuperar mejor sino **qué hacer con lo que ya se recuperó, turno tras turno**.

Esta sección no enseña nada nuevo: arma la herramienta. Si la clase de RAG está fresca, alcanza con correrla.

In [12]:
import pandas as pd
from huggingface_hub import hf_hub_download

# El mismo corpus de la clase de RAG, reconstruido con los mismos parametros.
REPO_HF, COMMIT_HF = "coastalcph/multi_eurlex", "6278c7994cd86297e823becad7fdf897faaea174"
PISO, TECHO, MIN_ART, PRESUPUESTO_CHARS, SEMILLA = 8_000, 60_000, 3, 40_000_000, 1337
RE_ARTICULO = re.compile(r"^[ \t]*Art[ií]culo\s+\d+", re.MULTILINE | re.IGNORECASE)

def reconstruir_corpus():
    marcos = []
    for split in ("validation", "test"):
        for rev in (COMMIT_HF, "refs/convert/parquet"):
            try:
                marcos.append(pd.read_parquet(hf_hub_download(
                    REPO_HF, "es/%s/0000.parquet" % split, repo_type="dataset", revision=rev)))
                break
            except Exception:
                continue
    if not marcos:
        raise RuntimeError("No se pudo bajar el corpus del espejo parquet.")
    d = pd.concat(marcos, ignore_index=True)
    d["largo"] = d["text"].str.len()
    d["articulos"] = d["text"].map(lambda t: len(RE_ARTICULO.findall(t)))
    d = d[(d.largo >= PISO) & (d.largo <= TECHO) & (d.articulos >= MIN_ART)]
    d = d.sample(frac=1.0, random_state=SEMILLA).reset_index(drop=True)
    return d[d["largo"].cumsum() <= PRESUPUESTO_CHARS].reset_index(drop=True)

corpus = reconstruir_corpus().head(N_ACTOS).reset_index(drop=True)
corpus["celex_id"] = corpus["celex_id"].astype(str)
TEXTO_DE = dict(zip(corpus.celex_id, corpus.text))

RE_FECHA = re.compile(r"^de\s+\d{1,2}\s+de\s+\w+", re.IGNORECASE)

def titulo_informativo(texto, tope=160):
    """El tipo de acto MAS su clausula descriptiva, salteando la linea de la fecha."""
    lineas = [l for l in (re.sub(r"[ \t]+", " ", l).strip() for l in texto.split("\n")) if l]
    if not lineas:
        return ""
    tema = next((l for l in lineas[1:6] if len(l) >= 40 and not RE_FECHA.match(l)), "")
    return (lineas[0] + (" — " + tema if tema else ""))[:tope]

TITULO_DE = {c: titulo_informativo(t) for c, t in TEXTO_DE.items()}

print("actos      :", len(corpus))
print("caracteres : %.1f M" % (corpus.largo.sum() / 1e6))
print("primero    :", corpus.celex_id.iloc[0], "— el mismo que en la clase de RAG")

es/validation/0000.parquet: reconstructing file:   0%|          |  0.00B / 19.0MB            

es/validation/0000.parquet: downloading bytes:           |  0.00B            

es/test/0000.parquet: reconstructing file:   0%|          |  0.00B / 27.4MB            

es/test/0000.parquet: downloading bytes:           |  0.00B            

actos      : 1544
caracteres : 30.4 M
primero    : 32015R0009 — el mismo que en la clase de RAG


In [13]:
from sentence_transformers import SentenceTransformer

embedder = SentenceTransformer("BAAI/bge-m3", device=DISPOSITIVO)
if DISPOSITIVO == "cuda":
    embedder.half()
# El mismo tope que en RAG: bge-m3 declara 8.192 tokens, el pipeline usa 1.024. Todo lo que
# pase de ahi se corta EN SILENCIO, por la derecha. La seccion 3 lo vuelve a necesitar.
embedder.max_seq_length = 1024
TOK_EMB = embedder.tokenizer

def embeber(textos, lote=16):
    return embedder.encode(list(textos), batch_size=lote, normalize_embeddings=True,
                           show_progress_bar=len(textos) > 400).astype(np.float32)

def contar_emb(texto):
    """Tokens de un texto segun el EMBEDDER. No es la misma unidad que contar()."""
    return len(TOK_EMB.encode(texto, add_special_tokens=False))

embeber(["prueba"])
vram("con el embedder cargado")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/15.8k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/687 [00:00<?, ?B/s]

pytorch_model.bin: reconstructing file:   0%|          |  0.00B / 2.27GB            

... (8 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

   [vram] con el embedder cargado    libre   5962 MiB · PyTorch   2.8 GB


In [14]:
def partir_en_articulos(texto):
    """[(etiqueta, cuerpo), ...] cortando por 'Articulo N' en linea propia."""
    cortes = [m.start() for m in RE_ARTICULO.finditer(texto)]
    if not cortes:
        return [("encabezado", texto)]
    partes = [("encabezado", texto[:cortes[0]])] if cortes[0] > 0 else []
    for i, ini in enumerate(cortes):
        fin = cortes[i + 1] if i + 1 < len(cortes) else len(texto)
        etiqueta = re.sub(r"\s+", " ", texto[ini:fin].split("\n")[0]).strip()
        partes.append((etiqueta, texto[ini:fin]))
    return partes

def partir_por_tokens(texto, tam=TAM_PEDAZO, solape=SOLAPE):
    ids = TOK_EMB.encode(texto, add_special_tokens=False)
    pedazos, paso = [], max(1, tam - solape)
    for i in range(0, len(ids), paso):
        trozo = ids[i:i + tam]
        if len(trozo) < 20 and pedazos:
            break
        pedazos.append(TOK_EMB.decode(trozo))
    return pedazos

def pedazos_de(texto, celex):
    """Corte por estructura: un articulo que entra va entero; uno largo se parte en tokens.
    Cada pedazo lleva adelante el titulo del acto, como en la clase de RAG."""
    salida = []
    for etiqueta, cuerpo in partir_en_articulos(texto):
        trozos = [cuerpo] if contar_emb(cuerpo) <= TAM_PEDAZO else partir_por_tokens(cuerpo)
        for tr in trozos:
            seccion = "" if etiqueta == "encabezado" else " · " + etiqueta
            salida.append({"id": "%s:%d" % (celex, len(salida)), "celex": celex,
                           "seccion": etiqueta, "crudo": tr,
                           "texto": "[%s%s] %s" % (recortar(TITULO_DE[celex], 140), seccion, tr)})
    return salida

t0 = time.time()
PEDAZOS = [p for c, t in TEXTO_DE.items() for p in pedazos_de(t, c)]
print("pedazos:", len(PEDAZOS), "· cortados en %.0f s" % (time.time() - t0))

t0 = time.time()
MATRIZ = embeber([p["texto"] for p in PEDAZOS])
print("índice denso listo en %.0f s" % (time.time() - t0))
vram("con el índice")

pedazos: 30730 · cortados en 38 s


Batches:   0%|          | 0/1921 [00:00<?, ?it/s]

índice denso listo en 367 s
   [vram] con el índice              libre  12100 MiB · PyTorch   2.8 GB


In [15]:
import logging
import bm25s, Stemmer
logging.getLogger("bm25s").setLevel(logging.WARNING)

# Las citas normativas sobreviven enteras a la tokenizacion: 1256/2011 -> n_1256_2011.
RE_ID_NORM = re.compile(r"(\d{1,5})/(\d{1,5})/(CEE|CE|UE|CECA|Euratom)", re.IGNORECASE)
RE_ID_NUM  = re.compile(r"(?<![^\W\d_])n\.?[oº°]?\s*(\d{1,5})/(\d{2,4})", re.IGNORECASE)

def normalizar_citas(t):
    t = RE_ID_NORM.sub(lambda m: "%s_%s_%s" % (m.group(1), m.group(2), m.group(3).upper()), t)
    return RE_ID_NUM.sub(lambda m: "n_%s_%s" % (m.group(1), m.group(2)), t)

stemmer = Stemmer.Stemmer("spanish")

def tokenizar(textos):
    return bm25s.tokenize([normalizar_citas(t) for t in textos],
                          stopwords="es", stemmer=stemmer, show_progress=False)

bm25_idx = bm25s.BM25()
bm25_idx.index(tokenizar([p["texto"] for p in PEDAZOS]), show_progress=False)

def buscar_bm25(consulta, n):
    idx, _ = bm25_idx.retrieve(tokenizar([consulta]), k=min(n, len(PEDAZOS)),
                               show_progress=False)
    return [PEDAZOS[int(i)] for i in idx[0]]

def rrf(listas, n, k=60):
    """Reciprocal Rank Fusion: cada lista vota con 1/(k + puesto)."""
    puntaje, por_id = {}, {}
    for lista in listas:
        for puesto, p in enumerate(lista, 1):
            puntaje[p["id"]] = puntaje.get(p["id"], 0.0) + 1.0 / (k + puesto)
            por_id[p["id"]] = p
    return [por_id[i] for i in sorted(puntaje, key=lambda i: -puntaje[i])[:n]]

def buscar(consulta, k=K):
    """El buscador de la clase de RAG: denso + lexico, fusionados por puesto.
    OJO: el canal denso trunca la consulta a los 1.024 tokens del embedder; el lexico, no."""
    q = embeber([consulta])[0]
    densos = [PEDAZOS[int(i)] for i in np.argsort(MATRIZ @ q)[::-1][:N_CANDIDATOS]]
    return rrf([densos, buscar_bm25(consulta, N_CANDIDATOS)], k)

def a_documentos(pasajes):
    """Los actos de los que salieron los pasajes, sin repetir y en orden de llegada."""
    vistos = []
    for p in pasajes:
        if p["celex"] not in vistos:
            vistos.append(p["celex"])
    return vistos

def mostrar_top(pasajes, gold=None, titulo=None):
    if titulo:
        print(titulo)
        print("─" * 74)
    for i, p in enumerate(pasajes, 1):
        marca = "   ← el correcto" if gold and p["celex"] == gold else ""
        print("%d. %-11s %-13s%s" % (i, p["celex"], recortar(p["seccion"], 13), marca))
        print("   " + recortar(TITULO_DE.get(p["celex"], ""), 110))
    print()

mostrar_top(buscar("posibilidades de pesca del Mar Báltico para 2012", 3),
            gold="32011R1256", titulo="prueba del buscador")

prueba del buscador
──────────────────────────────────────────────────────────────────────────
1. 32011R1256  Artículo 1      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
2. 32012R1088  Artículo 1   
   REGLAMENTO (UE) No 1088/2012 DEL CONSEJO — por el que se establecen, para 2013, las posibilidades de pesca apl…
3. 32010R1124  Artículo 1   
   REGLAMENTO (UE) No 1124/2010 DEL CONSEJO — por el que se establecen, para 2011, las posibilidades de pesca apl…



**Y el generador, conversando.** Hasta la clase de RAG, el prompt era uno: instrucciones, pasajes, pregunta. Ahora es una **lista de mensajes**: el de sistema con las instrucciones, después los turnos anteriores, y al final el turno de ahora con sus pasajes. Dos decisiones que la clase discute más adelante y que acá quedan escritas:

- La respuesta sale con el mismo schema de RAG (análisis, si alcanza el contexto, cita, respuesta). **Al historial va sólo lo que vio el usuario**: la respuesta y su cita. Ni el análisis ni los pasajes.
- Los pasajes viajan en el último mensaje, y sólo en ése. En el turno siguiente se recuperan otra vez.

In [16]:
from pydantic import BaseModel, Field
from typing import Literal

SISTEMA = ("Sos un asistente que responde preguntas sobre normativa europea, en una conversación. "
           "Respondé con los pasajes del <contexto> del último mensaje y con lo que se dijo en la "
           "conversación; no uses conocimiento propio. Si no alcanza, decilo. En cita van los "
           "identificadores entre corchetes de los pasajes que usaste. Respondé en dos o tres "
           "oraciones. Si el usuario sólo te cuenta algo, acusá recibo en una oración.")

class RespuestaChat(BaseModel):
    # Los campos van en orden alfabetico Y en el orden en que se piensan: el analisis antes
    # de la respuesta. Asi el orden es el mismo lo serialice como lo serialice el servidor.
    analisis: str = Field(description="Qué dicen el contexto y la conversación sobre la pregunta, en una oración.")
    basta_el_contexto: bool = Field(description="True si el contexto y la conversación alcanzan para responder.")
    cita: list[str] = Field(description="Identificadores entre corchetes de los pasajes usados.")
    respuesta: str = Field(description="La respuesta para el usuario.")

def armar_contexto(pasajes):
    """Cada pasaje con su identificador y, adentro, el titulo del acto adelante, como en la clase
    de RAG: un articulo suelto dice «el presente Reglamento» y no dice cual es."""
    return "\n\n".join("[%s · %s]\n%s" % (p["celex"], recortar(p["seccion"], 30), p["texto"])
                       for p in pasajes)

def mensaje_usuario(pregunta, pasajes):
    """El turno de ahora: con sus pasajes si los hay, o la pregunta sola."""
    if not pasajes:
        return pregunta
    return "<contexto>\n%s\n</contexto>\n\nPregunta: %s" % (armar_contexto(pasajes), pregunta)

def responder(pregunta, historial=(), consulta=None, pasajes=None, sistema=SISTEMA,
              num_ctx=NUM_CTX):
    """Un turno de la conversacion. consulta: con que se BUSCA (por default, la pregunta).
    pasajes=[] es un turno que no pregunta nada (el usuario cuenta algo): no se busca, y se
    contesta en texto, sin el schema de RAG, que lo empujaria a inventar un analisis."""
    if pasajes is None:
        pasajes = buscar(consulta or pregunta)
    msgs = [{"role": "system", "content": sistema}] + list(historial) + \
           [{"role": "user", "content": mensaje_usuario(pregunta, pasajes)}]
    if not pasajes:
        r = chat(msgs, num_predict=120, num_ctx=num_ctx)
        return dict(analisis="", basta_el_contexto=True, cita=[], respuesta=r["texto"] or (r["error"] or ""),
                    pasajes=[], mensajes=msgs, llamada=r)
    obj, r = estructurado(msgs, RespuestaChat, num_predict=800, num_ctx=num_ctx)
    if obj is None:
        obj = RespuestaChat(analisis="(no parseó)", basta_el_contexto=False, cita=[],
                            respuesta=solo_respuesta(None, r))
    return dict(obj.model_dump(), pasajes=pasajes, mensajes=msgs, llamada=r)

def limpiar_cortado(texto, tope=500):
    """Lo rescatado de un JSON cortado, sin la cola: un modelo chico atascado repite «} } }»
    hasta agotar los tokens, y esa cola NO puede entrar al historial (lo infla turno tras turno)."""
    texto = re.sub(r"(\s*[}\"]\s*){3,}.*$", "", texto, flags=re.S).strip()
    return recortar(texto, tope)

def solo_respuesta(obj, r):
    """El texto de la respuesta, aunque el JSON haya llegado cortado: si el modelo se quedó sin
    tokens a mitad del schema, se rescata el campo 'respuesta' en vez de imprimir el JSON crudo."""
    if obj is not None:
        return obj.respuesta
    m = re.search(r'"respuesta"\s*:\s*"((?:[^"\\]|\\.)*)', r["texto"] or "")
    if m:
        return limpiar_cortado(m.group(1))
    m = re.search(r'"analisis"\s*:\s*"((?:[^"\\]|\\.)*)', r["texto"] or "")
    if m:
        return "(JSON cortado; su análisis:) " + limpiar_cortado(m.group(1))
    return limpiar_cortado(r["texto"] or r["error"] or "", 400)

RE_CITA_ID = re.compile(r"\d{5}[A-Z]\d{4}[^\]\n]{0,40}")

def solo_citas(citas):
    """Las citas como [CELEX · seccion]. Un modelo chico a veces mete el pasaje entero en el
    campo de cita, y eso no tiene que entrar al historial."""
    salida = []
    for c in citas:
        m = RE_CITA_ID.search(c)
        salida.append("[%s]" % m.group(0).strip() if m else recortar(c, 50))
    return salida

def al_historial(historial, pregunta, salida):
    """Al historial va lo que se DIJO: la pregunta y la respuesta con su cita."""
    historial.append({"role": "user", "content": pregunta})
    cita = " ".join(solo_citas(salida["cita"]))
    historial.append({"role": "assistant",
                      "content": salida["respuesta"] + (" " + cita if cita else "")})

def texto_de(historial):
    """El historial como texto corrido, para leerlo o para buscar con el."""
    return "\n".join("%s: %s" % ("Usuario" if m["role"] == "user" else "Asistente", m["content"])
                     for m in historial)

> **Checkpoint.** Están el generador con sus contadores, el tokenizer que cuenta como el servidor, el corpus de la clase de RAG, el buscador híbrido y un turno de conversación. De acá en adelante, una conversación.

## 1 · El modelo no recuerda

*Momento 1 · La siguiente pregunta*

Hasta acá, cada pregunta fue la primera: el pipeline de RAG recibe una pregunta, recupera, responde con cita y se olvida de todo. Hoy, la segunda.

Primero, el guion: la conversación de la lámina 3. Es **una sola conversación** que corre por todo el notebook: las preguntas del usuario están escritas a mano y las respuestas las escribe el pipeline, turno a turno. Tiene tres partes, y cada una sirve a una sección distinta:

- **Un turno de apertura** con dos datos del usuario: dónde trabaja y cómo quiere las respuestas. Son los que se van a perder cuando la conversación crezca (secciones 4 y 6).
- **Seis turnos sobre pesca**, casi todos dependientes del anterior: elipsis, referencias, un cambio de año y un cambio de tema (sección 2). Cada uno lleva el acto que lo responde y una reescritura autónoma hecha a mano.
- **Un dato para el informe** —un reglamento y un artículo— y **trece preguntas** sacadas del eval set de la clase de RAG, para que la conversación llegue al turno 20 (secciones 3 y 8).

> **Sobre el acto del Báltico.** El reglamento que responde los primeros turnos, `32011R1256`, fija las posibilidades de pesca de 2012, pero **las cifras viven en sus anexos, y el corpus trae sólo los artículos**. Por eso el guion no pregunta cuánto se podía pescar: pregunta qué norma lo fijó y qué dice sobre el bacalao, que sí está en el texto.

In [17]:
# Trece preguntas del eval set de la clase de RAG, con su acto. Dejamos afuera las de pesca:
# la parte de pesca de la conversacion ya esta en los turnos 1 a 6.
RELLENO = [
    ("¿Qué reglas comunes rigen para que un avión matriculado en un país de la Unión siga "
     "estando en condiciones de volar?", "32014R1321"),
    ("¿Qué instrumento permite que una patente tenga efecto uniforme en varios países sin "
     "tener que registrarla en cada uno?", "32012R1257"),
    ("¿Qué condiciones hay que cumplir para afirmar en una etiqueta que un alimento ayuda a "
     "bajar el colesterol?", "32014R0686"),
    ("¿Qué ventaja aduanera tienen las mercancías que se importan en las islas atlánticas "
     "portuguesas, y durante cuánto tiempo rige?", "32010R0973"),
    ("¿Qué norma de 2012 amplió la lista de sustancias autorizadas en productos biocidas, y "
     "desde cuándo debían aplicarse sus disposiciones?", "32012L0002"),
    ("¿Qué se resolvió sobre la deuda que una empresa búlgara mantenía con el Estado sin que "
     "nadie la ejecutara durante años?", "32012D0706"),
    ("¿Qué se decidió sobre la venta en Europa de un algodón modificado genéticamente, y qué "
     "código único se le asignó?", "32011D0354"),
    ("¿Qué gravamen pagan los radiadores de aluminio que llegan desde China, y por qué se lo "
     "impuso?", "32012R1039"),
    ("¿Cómo se le cobra a los trenes de carga por el ruido que producen, y quién fija ese "
     "importe?", "32015R0429"),
    ("¿Con qué métodos de laboratorio se comprueba si un aceite de oliva cumple lo que "
     "declara su etiqueta?", "32013R1348"),
    ("¿En qué programa conjunto de investigación sobre mediciones participa la Unión junto a "
     "una treintena de países?", "32014D0555"),
    ("¿Qué protección obtuvo un dulce típico checo, y qué pasó con las marcas alemanas que "
     "usaban un nombre parecido?", "32011R0744"),
    ("¿Qué programa de financiación sostuvo entre 2014 y 2020 las acciones para construir un "
     "espacio común de justicia?", "32013R1382"),
]

# Cada turno: el texto, como depende de lo anterior, el acto que lo responde (gold) y una
# reescritura autonoma hecha a mano (humana), que es como lo buscaria una persona.
GUION = [
    {"turno": "Antes de empezar, dos cosas: trabajo en una pesquera de Vigo, y necesito las "
              "respuestas siempre con el artículo citado.", "tipo": "dato"},
    {"turno": "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?",
     "tipo": "autónoma", "gold": "32011R1256",
     "humana": "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?"},
    {"turno": "¿Y qué dice sobre el bacalao?", "tipo": "elipsis", "gold": "32011R1256",
     "humana": "¿Qué establece sobre el bacalao el reglamento de posibilidades de pesca del "
               "Mar Báltico para 2012?"},
    {"turno": "¿Ese reglamento cambió algo de la norma del año anterior?", "tipo": "referencia",
     "gold": "32011R1256",
     "humana": "¿Qué modificó el reglamento de posibilidades de pesca del Mar Báltico para 2012 "
               "del reglamento del año anterior?"},
    {"turno": "¿Y para 2013?", "tipo": "continuidad", "gold": "32012R1088",
     "humana": "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2013?"},
    {"turno": "Pasando al Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para otro?",
     "tipo": "cambio de tema", "gold": "32010R0724",
     "humana": "¿Cuándo se puede cerrar en tiempo real una pesquería en el Mar del Norte?"},
    {"turno": "¿A qué especies alcanza esa medida?", "tipo": "referencia", "gold": "32010R0724",
     "humana": "¿A qué especies se aplican los cierres en tiempo real de pesquerías en el Mar "
               "del Norte?"},
    {"turno": "Anotá esto para el informe: el que voy a citar es el Reglamento (UE) nº 1256/2011, "
              "artículo 9.", "tipo": "dato"},
] + [{"turno": p, "tipo": "autónoma", "gold": g} for p, g in RELLENO]

# Validacion blanda: que cada acto del guion este en el corpus que toco.
faltan = sorted({g["gold"] for g in GUION if g.get("gold") and g["gold"] not in TEXTO_DE})
print("turnos del guion:", len(GUION), "(del 0 al %d)" % (len(GUION) - 1))
if faltan:
    print("⚠  actos del guion que no están en este corpus:", ", ".join(faltan))
    print("   (con SMOKE_TEST = True falta el reglamento del Báltico de 2013; esos turnos no")
    print("    pueden acertar con ningún método, y la tabla de la sección 2 lo marca)")
for i, g in enumerate(GUION[:8]):
    print("  %2d  %-15s %s" % (i, g["tipo"], recortar(g["turno"], 80)))
print("  8–20  trece preguntas del eval set de RAG")

turnos del guion: 21 (del 0 al 20)
   0  dato            Antes de empezar, dos cosas: trabajo en una pesquera de Vigo, y necesito las res…
   1  autónoma        ¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?
   2  elipsis         ¿Y qué dice sobre el bacalao?
   3  referencia      ¿Ese reglamento cambió algo de la norma del año anterior?
   4  continuidad     ¿Y para 2013?
   5  cambio de tema  Pasando al Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para o…
   6  referencia      ¿A qué especies alcanza esa medida?
   7  dato            Anotá esto para el informe: el que voy a citar es el Reglamento (UE) nº 1256/201…
  8–20  trece preguntas del eval set de RAG


### 1.1 · La siguiente pregunta

En Claude, una segunda pregunta como «¿y qué dice sobre el bacalao?» anda. Acá, el pipeline de RAG, tal cual, contestando los turnos 1 y 2 **por separado**: cada llamada sin historial, como en la clase pasada.

In [18]:
t1, t2 = GUION[1], GUION[2]
s1 = responder(t1["turno"])     # sin historial: el pipeline de la clase de RAG
s2 = responder(t2["turno"])     # el turno 2, tambien sin historial

mostrar("TURNO 1 · " + t1["turno"], "%s\ncita: %s" % (s1["respuesta"], " ".join(s1["cita"]) or "—"))
mostrar("TURNO 2 · " + t2["turno"] + "   (sin historial)",
        "%s\ncita: %s" % (s2["respuesta"], " ".join(s2["cita"]) or "—"))
mostrar_top(s2["pasajes"], gold=t2["gold"],
            titulo="LO QUE TRAJO EL BUSCADOR PARA «%s»" % t2["turno"])
RESULTADOS["1 · dos turnos sin historial"] = {
    "turno 1": recortar(s1["respuesta"], 200), "cita 1": s1["cita"],
    "turno 2": recortar(s2["respuesta"], 200),
    "top-5 turno 2": a_documentos(s2["pasajes"])}

──────────────────────────────────────────────────────────────────────────
TURNO 1 · ¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?
──────────────────────────────────────────────────────────────────────────
El Reglamento (UE) No 1256/2011 fijó las posibilidades de pesca del Mar
Báltico para 2012.
cita: [32011R1256 · Artículo 1] [32011R1256 · encabezado]

──────────────────────────────────────────────────────────────────────────
TURNO 2 · ¿Y qué dice sobre el bacalao?   (sin historial)
──────────────────────────────────────────────────────────────────────────
El contexto menciona que el porcentaje de capturas de bacalao no debe
superar el 1,5 % del total de capturas del grupo de buques, y que se puede
excluir a ciertos grupos de buques del régimen de gestión del esfuerzo
pesquero si se cumplen ciertas condiciones. Además, se aborda la gestión
del esfuerzo pesquero para el bacalao en el Mar Báltico, permitiendo
cierta flexibilidad en la asignación de cuotas y 

**Lo que hay que ver.** El turno 1 se sostiene solo: el buscador trae el reglamento de 2012 y el generador puede responder con su número y su artículo. El turno 2 falla **dos veces**, y son dos fallas distintas. El generador recibe una frase corta sin saber de qué se venía hablando: no hay mar, no hay año, no hay reglamento, ni siquiera sobre qué norma «dice». Y el buscador busca con esas mismas palabras, así que trae bacalao de cualquier lado. Las vamos a arreglar en ese orden: primero que el modelo sepa qué se dijo; después, en la sección 2, que el buscador sepa qué buscar.

### 1.2 · El modelo no recuerda: lee

Tres llamadas. En la primera el usuario se presenta. La segunda pregunta su nombre, sola. La tercera pregunta lo mismo, pero mandando la lista con los mensajes anteriores. Quien guarda esa lista y la vuelve a mandar no es el modelo: es el programa que lo rodea, el **harness**. En Claude lo escribió Anthropic; acá lo escribimos nosotros, y la lista es una variable de Python.

In [19]:
a  = chat([{"role": "user", "content": "Me llamo Ana."}], num_predict=60)
b  = chat([{"role": "user", "content": "¿Cómo me llamo?"}], num_predict=60)
b2 = chat([{"role": "user", "content": "Me llamo Ana."},
           {"role": "assistant", "content": a["texto"]},
           {"role": "user", "content": "¿Cómo me llamo?"}], num_predict=60)

print("LLAMADA A   «Me llamo Ana.»                        →", recortar(a["texto"], 80))
print("LLAMADA B   «¿Cómo me llamo?»                      →", recortar(b["texto"], 80))
print("LLAMADA B'  [Me llamo Ana · %s · ¿Cómo me llamo?]" % recortar(a["texto"], 20))
print("                                                   →", recortar(b2["texto"], 80))
print()
print("Entre A y B no hay nada que las una. La memoria es la lista, y la lista la tiene el harness: acá, este notebook.")
RESULTADOS["1 · sin estado"] = {"A": a["texto"], "B": b["texto"], "B'": b2["texto"]}

LLAMADA A   «Me llamo Ana.»                        → ¡Hola, Ana! ¿En qué puedo ayudarte? 😊
LLAMADA B   «¿Cómo me llamo?»                      → ¡Hola! Mi nombre es Qwen. ¿En qué puedo ayudarte?
LLAMADA B'  [Me llamo Ana · ¡Hola, Ana! ¿En qué … · ¿Cómo me llamo?]
                                                   → Te llamas Ana. 😊 ¿En qué puedo ayudarte hoy?

Entre A y B no hay nada que las una. La memoria es la lista, y la lista la tiene el harness: acá, este notebook.


### 1.3 · Un chat es una lista que crece

Lo que el harness manda es una lista de mensajes con roles. Pero el modelo no lee listas: la **plantilla de chat** (*chat template*) la convierte en un único texto, con tokens especiales que marcan dónde empieza y termina cada turno y de quién es. Abajo, la lista del turno 2 tal como la lee el modelo.

*Nota de laboratorio:* la plantilla es la de Qwen3 en HuggingFace. Ollama trae su propia copia, que hace lo mismo y agrega un `/no_think` al último turno del usuario.

In [20]:
lista_1 = [{"role": "system", "content": SISTEMA},
           {"role": "user", "content": t1["turno"]}]
lista_2 = lista_1 + [{"role": "assistant", "content": s1["respuesta"]},
                     {"role": "user", "content": t2["turno"]}]

renderizado = plantilla(lista_2)
print("LA LISTA DEL TURNO 2, COMO LA LEE EL MODELO")
print("─" * 74)
print(renderizado)
print("─" * 74)
especiales = re.findall(r"<\|[a-z_]+\|>|</?think>", renderizado)
print("tokens especiales:", ", ".join("%s ×%d" % (t, especiales.count(t))
                                      for t in dict.fromkeys(especiales)))
print()
print("tokens del prompt · turno 1: %d · turno 2: %d"
      % (contar_prompt(lista_1), contar_prompt(lista_2)))
print("El turno 2 lleva adentro el turno 1 entero. En la sección 3, el turno 20.")
RESULTADOS["1 · plantilla"] = {"turno 1": contar_prompt(lista_1), "turno 2": contar_prompt(lista_2)}

LA LISTA DEL TURNO 2, COMO LA LEE EL MODELO
──────────────────────────────────────────────────────────────────────────
<|im_start|>system
Sos un asistente que responde preguntas sobre normativa europea, en una conversación. Respondé con los pasajes del <contexto> del último mensaje y con lo que se dijo en la conversación; no uses conocimiento propio. Si no alcanza, decilo. En cita van los identificadores entre corchetes de los pasajes que usaste. Respondé en dos o tres oraciones. Si el usuario sólo te cuenta algo, acusá recibo en una oración.<|im_end|>
<|im_start|>user
¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?<|im_end|>
<|im_start|>assistant
El Reglamento (UE) No 1256/2011 fijó las posibilidades de pesca del Mar Báltico para 2012.<|im_end|>
<|im_start|>user
¿Y qué dice sobre el bacalao?<|im_end|>
<|im_start|>assistant
<think>

</think>


──────────────────────────────────────────────────────────────────────────
tokens especiales: <|im_start|> ×5, <|im_e

### 1.4 · Las dos memorias del modelo

Si el modelo no guarda nada entre llamadas, ¿de dónde sale lo que sí sabe? De dos lugares muy distintos, que la literatura separa con nombre propio:

- **Memoria paramétrica: los pesos.** Se escribieron una vez, en el entrenamiento, y no guardan el texto que el modelo leyó: guardan regularidades, lo que se repitió muchas veces. Por eso el modelo sabe lo frecuente y no lo raro, y lo raro lo completa con la misma fluidez.
- **Memoria en contexto: la ventana.** Lo que está en el prompt está entero, token por token, y el modelo lo puede citar tal cual. Pero dura una llamada.

Tres llamadas al mismo modelo, sin historial: una pregunta frecuente, una rara y la misma rara con la fuente en la ventana. La rara es el dato que el usuario va a anotar en el turno 7: el artículo 9 del Reglamento 1256/2011.

In [21]:
# Las dos memorias, en tres llamadas sin historial. La fuente es el articulo 9 tal como esta
# en el corpus: el mismo dato que el usuario va a anotar en el turno 7.
ART9 = next((cuerpo for et, cuerpo in partir_en_articulos(TEXTO_DE.get("32011R1256", ""))
             if re.match(r"art[ií]culo\s+9\b", et, re.I)), "")
PREGUNTA_ART9 = "¿Qué dice el artículo 9 del Reglamento (UE) nº 1256/2011? Respondé en dos oraciones."

frecuente = ask("¿Cuál es la capital de España? Respondé en una palabra.", num_predict=20)
de_memoria = ask(PREGUNTA_ART9, num_predict=200)
con_fuente = (ask("<contexto>\n%s\n</contexto>\n\n%s" % (ART9, PREGUNTA_ART9), num_predict=200)
              if ART9 else "(el artículo 9 no está en este corpus)")

mostrar("DESDE LOS PESOS · LO FRECUENTE · «¿Cuál es la capital de España?»", frecuente)
mostrar("DESDE LOS PESOS · LO RARO · «%s»" % PREGUNTA_ART9, de_memoria)
mostrar("CON LA FUENTE EN LA VENTANA · la misma pregunta", con_fuente)
mostrar("EL ARTÍCULO 9, COMO ESTÁ EN EL CORPUS", recortar(ART9, 500) or "—")
RESULTADOS["1 · dos memorias"] = {"frecuente": frecuente, "raro, de memoria": recortar(de_memoria, 200),
                                  "raro, con la fuente": recortar(con_fuente, 200)}

──────────────────────────────────────────────────────────────────────────
DESDE LOS PESOS · LO FRECUENTE · «¿Cuál es la capital de España?»
──────────────────────────────────────────────────────────────────────────
Madrid

──────────────────────────────────────────────────────────────────────────
DESDE LOS PESOS · LO RARO · «¿Qué dice el artículo 9 del Reglamento (UE) nº 1256/2011? Respondé en dos oraciones.»
──────────────────────────────────────────────────────────────────────────
El artículo 9 del Reglamento (UE) nº 1256/2011 establece las condiciones
para la aplicación de la regla de la doble imposición en el contexto de
los derechos de propiedad intelectual. Este artículo también define los
tipos de derechos de propiedad intelectual que están sujetos a este
régimen.

──────────────────────────────────────────────────────────────────────────
CON LA FUENTE EN LA VENTANA · la misma pregunta
──────────────────────────────────────────────────────────────────────────
El artículo 9 del 

**Cómo leerlo, salga como salga.** Lo frecuente sale de los pesos sin problema: lo leyeron miles de veces. Lo raro, no: sin la fuente, el modelo contesta de memoria —inventa con la misma fluidez, o admite que no sabe— y no puede citar el artículo. Con el artículo en la ventana, lo lee y lo cita. **Una memoria enorme que no se puede escribir, y una exacta que dura una llamada.** Todo lo que sigue es administrar la segunda.

Cuánto ocupa cada una, en bits, queda para la sección 3: primero hay que ver qué guarda la ventana por cada token, el KV cache.

▶ **Láminas 5–8** (la siguiente pregunta; el modelo no recuerda: lee; un chat es una lista que crece; las dos memorias) y la **3** (la conversación que vamos a seguir: el guion de arriba). La apertura (láminas 1, 2 y 4) y el mapa de la clase (9 y 10) no tienen celda.

## 2 · Buscar en una conversación

*Momento 1 · La siguiente pregunta (sigue)*

Hacia dónde vamos: el generador ya puede recibir el historial; falta el buscador. Primero arrancamos la conversación de verdad, con historial. El turno 0 no pregunta nada —es un dato—, así que no se busca: el modelo sólo acusa recibo.

In [22]:
HIST = []      # la conversacion: lo que se dijo, y nada mas
TURNOS = []    # por turno: la pregunta, con que se busco, que se trajo y que se respondio

def correr_turno(i, consulta=None):
    """Un turno del guion, con el historial que haya. Un turno 'dato' no busca."""
    g = GUION[i]
    if g["tipo"] == "dato":
        salida = responder(g["turno"], HIST, pasajes=[])
    else:
        salida = responder(g["turno"], HIST, consulta=consulta)
    TURNOS.append(dict(g, i=i, consulta=consulta or g["turno"], salida=salida))
    al_historial(HIST, g["turno"], salida)
    return salida

for i in (0, 1):
    s = correr_turno(i)
    print("turno %d · %s" % (i, GUION[i]["turno"]))
    print(parrafo(s["respuesta"] + "  " + " ".join(s["cita"])))
    print()

turno 0 · Antes de empezar, dos cosas: trabajo en una pesquera de Vigo, y necesito las respuestas siempre con el artículo citado.
   Acuso recibo. Estoy aquí para ayudarte con la normativa europea
   relacionada con tu trabajo en una pesquera de Vigo, citando siempre el
   artículo correspondiente. ¿En qué necesitas ayuda?

turno 1 · ¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?
   El Reglamento (UE) No 1256/2011 fijó las posibilidades de pesca del Mar
   Báltico para 2012. [32011R1256 · Artículo 1] y [32011R1256 · encabezado]
   32011R1256 · Artículo 1 32011R1256 · encabezado



### 2.1 · El buscador no vio el turno 1

El turno 2 otra vez, ahora **con el historial** para el generador. El buscador, en cambio, sigue recibiendo sólo el último turno.

In [23]:
g2 = GUION[2]
solo = responder(g2["turno"], HIST)          # el generador ve el historial; el buscador, no

mostrar("TURNO 2 CON HISTORIAL · el buscador recibe sólo «%s»" % g2["turno"],
        "%s\ncita: %s" % (solo["respuesta"], " ".join(solo["cita"]) or "—"))
mostrar_top(solo["pasajes"], gold=g2["gold"], titulo="TOP-5 DE «%s», BUSCADO SOLO" % g2["turno"].upper())
RESULTADOS["2 · turno 2 buscado solo"] = {"top-5": a_documentos(solo["pasajes"]),
                                          "respuesta": recortar(solo["respuesta"], 200)}

──────────────────────────────────────────────────────────────────────────
TURNO 2 CON HISTORIAL · el buscador recibe sólo «¿Y qué dice sobre el bacalao?»
──────────────────────────────────────────────────────────────────────────
(JSON cortado; su análisis:) El contexto menciona que el bacalao es una
especie clave en la regulación de las posibilidades de pesca en el Mar
Báltico y el Mar Céltico. Se establecen límites de captura y se menciona
la necesidad de gestionar el esfuerzo pesquero para evitar un aumento de
la mortalidad por pesca. Además, se refiere a la flexibilidad en la
gestión del esfuerzo pesquero para las poblaciones de bacalao del Mar
Báltico, permitiendo una asignación más eficaz de cuotas y reacciones
rápidas ante intercambios de cuo…
cita: —

TOP-5 DE «¿Y QUÉ DICE SOBRE EL BACALAO?», BUSCADO SOLO
──────────────────────────────────────────────────────────────────────────
1. 32010R0712  encabezado   
   REGLAMENTO (UE) No 712/2010 DEL CONSEJO — por el que se modifica el 

**El diagnóstico.** El generador entiende la pregunta —tiene el turno 1 delante— y la contesta con fluidez, con lo que le llegó. El buscador no: con una frase tan corta, el vector cae en cualquier pasaje sobre bacalao. El corpus tiene un reglamento de posibilidades de pesca del Báltico por año, casi idénticos entre sí, y otros actos de bacalao en otros mares. Son las dos superficies de falla de RAG: acá falla la recuperación, y no por un mal embedder, sino porque su entrada dejó de ser una pregunta.

### 2.2 · Tres formas de depender

No todas las preguntas dependen igual, y lo que arregla una puede romper otra. Los turnos dependientes del guion, con su tipo:

In [24]:
for i in range(2, 7):
    g = GUION[i]
    print("  turno %d · %-15s %s" % (i, g["tipo"], g["turno"]))
    print("            gold %s · a mano: %s" % (g["gold"], recortar(g["humana"], 70)))
print()
print("La elipsis omite lo que ya se dijo; la referencia lo nombra con otra palabra; la")
print("continuidad mantiene casi todo y cambia una cosa; el cambio de tema NO tiene que arrastrar")
print("el anterior. Arrastrar contexto ayuda a las tres primeras y arruina la última.")

  turno 2 · elipsis         ¿Y qué dice sobre el bacalao?
            gold 32011R1256 · a mano: ¿Qué establece sobre el bacalao el reglamento de posibilidades de pesc…
  turno 3 · referencia      ¿Ese reglamento cambió algo de la norma del año anterior?
            gold 32011R1256 · a mano: ¿Qué modificó el reglamento de posibilidades de pesca del Mar Báltico …
  turno 4 · continuidad     ¿Y para 2013?
            gold 32012R1088 · a mano: ¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2…
  turno 5 · cambio de tema  Pasando al Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para otro?
            gold 32010R0724 · a mano: ¿Cuándo se puede cerrar en tiempo real una pesquería en el Mar del Nor…
  turno 6 · referencia      ¿A qué especies alcanza esa medida?
            gold 32010R0724 · a mano: ¿A qué especies se aplican los cierres en tiempo real de pesquerías en…

La elipsis omite lo que ya se dijo; la referencia lo nombra con otra palabra; la
conti

### 2.3 · Buscar con toda la conversación

El arreglo obvio: que el buscador vea todo. Concatenamos la conversación entera, con el turno nuevo al final, y buscamos con eso.

In [25]:
consulta_entera = texto_de(HIST + [{"role": "user", "content": g2["turno"]}])
print("la conversación como consulta: %d tokens del embedder (el pipeline corta en %d)"
      % (contar_emb(consulta_entera), embedder.max_seq_length))
print()
mostrar_top(buscar(consulta_entera), gold=g2["gold"],
            titulo="TOP-5 DE LA CONVERSACIÓN ENTERA, EN EL TURNO 2")

la conversación como consulta: 168 tokens del embedder (el pipeline corta en 1024)

TOP-5 DE LA CONVERSACIÓN ENTERA, EN EL TURNO 2
──────────────────────────────────────────────────────────────────────────
1. 32011R1256  Artículo 1      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
2. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
3. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
4. 32011R1256  Artículo 2      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
5. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca

Puede acertar en el turno 2: la conversación todavía es corta y habla de una sola cosa. Lo que hay que mirar es qué pasa **turno tras turno**, y eso lo mide la vara de abajo. Dos de las tres razones por las que este arreglo pierde ya se pueden anticipar: un solo vector promedia toda la conversación, así que casi no se mueve de un turno al siguiente; y el embedder corta por la derecha, así que cuando la conversación es larga lo que se cae es justo el turno nuevo. La segunda la medimos en la sección 3, cuando la conversación ya es larga.

### 2.4 · Reescribir la pregunta

Antes de buscar, una llamada al modelo completa el turno: con los últimos mensajes delante, lo reescribe como una pregunta que se sostiene sola. Es la **reescritura de la consulta** (*query rewriting*, o *condense question*). La salida va restringida a un schema de un solo campo: no hay lugar para razonar en voz alta ni para contestar, sólo para escribir una consulta.

In [26]:
class Consulta(BaseModel):
    consulta: str = Field(description="El último mensaje del usuario, reescrito para que se entienda solo.")

PROMPT_REESCRIBIR = """Conversación hasta ahora:
{conversacion}

Último mensaje del usuario: {turno}

Reescribí el último mensaje del usuario como una consulta de búsqueda que se entienda sin la
conversación: completá lo que el mensaje da por sabido (de qué se habla, qué mar, qué año, qué
norma). No la respondas y no agregues datos que no estén en la conversación. Si el mensaje
cambia de tema, no arrastres el tema anterior."""

def reescribir(historial, turno, ultimos=6):
    """La consulta autonoma para el buscador. Mira los ultimos mensajes, no la conversacion entera."""
    if not historial:
        return turno
    obj, _ = estructurado([{"role": "user", "content": PROMPT_REESCRIBIR.format(
        conversacion=texto_de(historial[-ultimos:]), turno=turno)}], Consulta, num_predict=120)
    return obj.consulta.strip() if obj and obj.consulta.strip() else turno

r2 = reescribir(HIST, g2["turno"])
print("turno      :", g2["turno"])
print("reescritura:", r2)
print()
mostrar_top(buscar(r2), gold=g2["gold"], titulo="TOP-5 DE LA REESCRITURA")

turno      : ¿Y qué dice sobre el bacalao?
reescritura: ¿Qué dice el Reglamento (UE) No 1256/2011 sobre el bacalao en el Mar Báltico para 2012?

TOP-5 DE LA REESCRITURA
──────────────────────────────────────────────────────────────────────────
1. 32011R1256  Artículo 1      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
2. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
3. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
4. 32011R1256  encabezado      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen, para 2012, las posibilidades de pesca apl…
5. 32011R1256  Artículo 2      ← el correcto
   REGLAMENTO (UE) No 1256/2011 DEL CONSEJO — por el que se establecen,

**Desde acá, cada superficie recibe lo suyo:** el buscador, la reescritura; el generador, el historial y los pasajes. Ocupa el mismo lugar del pipeline que HyDE: una llamada antes de recuperar, en cada turno.

### 2.5 · La vara: cuatro formas de buscar, sobre los mismos turnos

La conversación sigue hasta el turno 6, y en cada turno dependiente buscamos de cuatro formas: el último turno solo, la conversación entera, la reescritura del modelo y la reescritura a mano. El turno se contesta con la reescritura del modelo —es el pipeline nuevo—, y las cuatro búsquedas se miden con la misma vara determinista de RAG: **¿el acto correcto está entre los cinco pasajes?** (hit@5).

In [27]:
MODOS = ["último turno", "conversación entera", "reescritura del modelo", "reescritura a mano"]
VARA = []

for i in range(2, 7):
    g = GUION[i]
    conversacion = texto_de(HIST)
    reesc = reescribir(HIST, g["turno"])
    consultas = {"último turno": g["turno"],
                 "conversación entera": texto_de(HIST + [{"role": "user", "content": g["turno"]}]),
                 "reescritura del modelo": reesc,
                 "reescritura a mano": g["humana"]}
    docs = {m: a_documentos(buscar(q)) for m, q in consultas.items()}
    VARA.append(dict(i=i, tipo=g["tipo"], gold=g["gold"], turno=g["turno"],
                     conversacion=conversacion, consultas=consultas, docs=docs))
    correr_turno(i, consulta=reesc)                 # el turno se contesta con el pipeline nuevo
    print("turno %d listo · %s" % (i, recortar(g["turno"], 60)))

turno 2 listo · ¿Y qué dice sobre el bacalao?
turno 3 listo · ¿Ese reglamento cambió algo de la norma del año anterior?
turno 4 listo · ¿Y para 2013?
turno 5 listo · Pasando al Mar del Norte: ¿cuándo se puede cerrar una pesque…
turno 6 listo · ¿A qué especies alcanza esa medida?


**Primero se leen, después se miden.** Un reescritor es un modelo, y falla como un modelo: la consulta sale prolija y busca otra cosa. Abajo, cada reescritura al lado de la escrita a mano, con dos banderas automáticas que no reemplazan leerlas: **eco** (devolvió el turno casi igual) y **números nuevos** (agregó un año o un número que nadie dijo).

In [28]:
def banderas(reesc, turno, conversacion):
    b = []
    if len(reesc.split()) <= len(turno.split()) + 2:
        b.append("eco")
    nuevos = set(re.findall(r"\d+", reesc)) - set(re.findall(r"\d+", conversacion + " " + turno))
    if nuevos:
        b.append("números nuevos: " + ", ".join(sorted(nuevos)))
    return b

for v in VARA:
    print("turno %d (%s) · «%s»" % (v["i"], v["tipo"], v["turno"]))
    print("   modelo :", v["consultas"]["reescritura del modelo"])
    print("   a mano :", v["consultas"]["reescritura a mano"])
    b = banderas(v["consultas"]["reescritura del modelo"], v["turno"], v["conversacion"])
    if b:
        print("   ⚑", " · ".join(b))
    print()

turno 2 (elipsis) · «¿Y qué dice sobre el bacalao?»
   modelo : ¿Qué dice el Reglamento (UE) No 1256/2011 sobre el bacalao en el Mar Báltico para 2012?
   a mano : ¿Qué establece sobre el bacalao el reglamento de posibilidades de pesca del Mar Báltico para 2012?

turno 3 (referencia) · «¿Ese reglamento cambió algo de la norma del año anterior?»
   modelo : ¿El Reglamento (UE) No 1256/2011 cambió algo respecto a la norma del año anterior en el Mar Báltico?
   a mano : ¿Qué modificó el reglamento de posibilidades de pesca del Mar Báltico para 2012 del reglamento del año anterior?

turno 4 (continuidad) · «¿Y para 2013?»
   modelo : ¿Cuál fue el reglamento que estableció las posibilidades de pesca para el Mar Báltico en el año 2013?
   a mano : ¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2013?

turno 5 (cambio de tema) · «Pasando al Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para otro?»
   modelo : cuándo se puede cerrar una pesquería de un día

In [29]:
TIPOS = ["elipsis", "referencia", "continuidad", "cambio de tema"]

def repetidos(modo):
    """Cuantos de los actos traidos se repiten respecto del turno anterior, en promedio."""
    pares = [len(set(VARA[j]["docs"][modo]) & set(VARA[j - 1]["docs"][modo]))
             for j in range(1, len(VARA))]
    return "%.1f de %d" % (np.mean(pares), K) if pares else "—"

filas = []
for m in MODOS:
    fila = {}
    for t in TIPOS:
        sub = [v for v in VARA if v["tipo"] == t]
        fila[t] = "%d/%d" % (sum(v["gold"] in v["docs"][m] for v in sub), len(sub))
    fila["total"] = "%d/%d" % (sum(v["gold"] in v["docs"][m] for v in VARA), len(VARA))
    fila["repite del turno anterior"] = repetidos(m)
    filas.append((m, fila))
tabla("hit@5 · el acto correcto entre los cinco pasajes, en los turnos dependientes",
      TIPOS + ["total", "repite del turno anterior"], filas)

print("turno por turno:")
for v in VARA:
    print("  %d %-15s %s" % (v["i"], v["tipo"],
          "  ".join("%s %s" % ("✓" if v["gold"] in v["docs"][m] else "✗", m) for m in MODOS)))
if any(v["gold"] not in TEXTO_DE for v in VARA):
    print("\n  ⚠ hay turnos cuyo acto no está en este corpus: ningún método puede acertarlos.")
RESULTADOS["2 · la vara de la reescritura"] = dict(filas)


hit@5 · el acto correcto entre los cinco pasajes, en los turnos dependientes
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                           elipsis   referencia   continuidad   cambio de tema   total   repite del turno anterior
último turno                   1/1          0/2           0/1              1/1     2/5                    0.0 de 5
conversación entera            1/1          1/2           1/1              0/1     3/5                    1.8 de 5
reescritura del modelo         1/1          2/2           1/1              1/1     5/5                    0.8 de 5
reescritura a mano             1/1          2/2           1/1              1/1     5/5                    1.2 de 5

turno por turno:
  2 elipsis         ✓ último turno  ✓ conversación entera  ✓ reescritura del modelo  ✓ reescritura a mano
  3 referencia      ✗ último turno  ✓ conversación entera  ✓ reescritura del modelo  ✓ reescritura a mano
  

**Cómo leerlo, salga como salga.** Son cinco turnos de una conversación: es ilustrativo, no estadístico, y la versión con muchas conversaciones es parte del segundo proyecto. Tres cosas para mirar:

- **La columna de repetidos.** Si la conversación entera trae casi los mismos actos turno tras turno, es el promedio que no se mueve: la pregunta de ahora pesa una fracción del vector.
- **Reescritura del modelo contra la escrita a mano.** La mejor reescritura no es la que mejor se lee: es la que trae el pasaje correcto. A veces la del humano pierde, porque es más corta y le da menos de dónde agarrarse al canal léxico.
- **El cambio de tema.** Es el único tipo donde arrastrar la conversación es un error. Si la reescritura del modelo lo trae con el Báltico adentro, falló en corrección aunque acierte.

> **Estamos acá · fin del momento 1, la siguiente pregunta.** **Resuelto:** el modelo ve la conversación —el harness le re-envía la lista entera en cada llamada— y el buscador recibe la pregunta reescrita, que se sostiene sola. **Costó:** una llamada más por turno, la del reescritor, y una falla nueva que no avisa: una reescritura prolija que busca otra cosa. **El problema nuevo:** el reescritor y el generador leen el historial, y el historial viaja entero en cada turno. En el turno 20, cada llamada carga los veinte anteriores.

▶ **Láminas 11–17** (el buscador no vio el turno 1; tres formas de depender; buscar con toda la conversación; reescribir; cómo se equivoca un reescritor; la vara; estamos acá). El punto de corte del embedder está en la sección 3.

## 3 · Mandar todo cuesta

*Momento 2 · La conversación larga*

Ya sabemos con qué buscar en cada turno. Ahora, lo que se manda: la conversación entera, otra vez, cada vez. Seguimos el guion hasta el turno 20 con el pipeline nuevo —reescribir, buscar, responder con el historial entero— y la política más simple de todas: **si entra, se pega**. El turno 7 es el dato para el informe; del 8 al 20, preguntas del eval set de RAG.

In [30]:
t0 = time.time()
for i in range(7, len(GUION)):
    g = GUION[i]
    consulta = None if g["tipo"] == "dato" else reescribir(HIST, g["turno"])
    s = correr_turno(i, consulta)
    if s["llamada"]["error"]:
        print("turno %2d · ⚠ el servidor no contestó (%s) · %s" % (i, recortar(s["llamada"]["error"], 60), recortar(g["turno"], 40)))
    else:
        print("turno %2d · prompt %5d tokens · %s" % (i, s["llamada"]["prompt"], recortar(g["turno"], 56)))
print("\nlos turnos 7 a %d, en %.0f s" % (len(GUION) - 1, time.time() - t0))

turno  7 · prompt  1409 tokens · Anotá esto para el informe: el que voy a citar es el Reg…
turno  8 · prompt  3524 tokens · ¿Qué reglas comunes rigen para que un avión matriculado …
turno  9 · prompt  3872 tokens · ¿Qué instrumento permite que una patente tenga efecto un…
turno 10 · prompt  4593 tokens · ¿Qué condiciones hay que cumplir para afirmar en una eti…
turno 11 · prompt  3876 tokens · ¿Qué ventaja aduanera tienen las mercancías que se impor…
turno 12 · prompt  5185 tokens · ¿Qué norma de 2012 amplió la lista de sustancias autoriz…
turno 13 · prompt  5202 tokens · ¿Qué se resolvió sobre la deuda que una empresa búlgara …
turno 14 · prompt  4817 tokens · ¿Qué se decidió sobre la venta en Europa de un algodón m…
turno 15 · prompt  5575 tokens · ¿Qué gravamen pagan los radiadores de aluminio que llega…
turno 16 · prompt  5375 tokens · ¿Cómo se le cobra a los trenes de carga por el ruido que…
turno 17 · prompt  5543 tokens · ¿Con qué métodos de laboratorio se comprueba si un aceit…

### 3.1 · El turno 20 manda veinte turnos

Por turno: cuánto del prompt es **historial re-enviado**, cuánto mide el prompt entero —con sus pasajes—, cuánto dice el servidor que leyó, y el **acumulado**: lo que la conversación procesó en total hasta ese turno.

In [31]:
COSTO, filas, acumulado = [], [], 0
for t in TURNOS:
    msgs = t["salida"]["mensajes"]
    total = contar_prompt(msgs)
    sin_historial = contar_prompt([msgs[0], msgs[-1]])
    historial = total - sin_historial
    acumulado += total
    COSTO.append(dict(i=t["i"], total=total, historial=historial,
                      servidor=t["salida"]["llamada"]["prompt"], acumulado=acumulado))
    filas.append(("turno %d" % t["i"], {"historial re-enviado": historial, "prompt entero": total,
                                        "según el servidor": t["salida"]["llamada"]["prompt"] or "—",
                                        "acumulado": acumulado}))
tabla("Lo que se manda en cada turno, en tokens del generador",
      ["historial re-enviado", "prompt entero", "según el servidor", "acumulado"], filas)

u = COSTO[-1]
print("El turno %d re-envía %d tokens de historial y manda %d en total." % (u["i"], u["historial"], u["total"]))
print("Para llegar ahí, la conversación ya procesó %d tokens: %.1f veces el largo de su último prompt."
      % (u["acumulado"], u["acumulado"] / u["total"]))
print("Una conversación no cuesta lo que mide: cuesta la suma de todos sus prefijos.")
RESULTADOS["3 · costo"] = {"turno": u["i"], "historial": u["historial"], "prompt": u["total"],
                           "acumulado": u["acumulado"]}


Lo que se manda en cada turno, en tokens del generador
───────────────────────────────────────────────────────────────────────────────────
               historial re-enviado   prompt entero   según el servidor   acumulado
turno 0                           0             160                 160         160
turno 1                          88            1736                1736        1896
turno 2                         230            1952                1952        3848
turno 3                         385            2032                2032        5880
turno 4                         559            2174                2174        8054
turno 5                         793            2876                2876       10930
turno 6                         964            2786                2786       13716
turno 7                        1242            1409                1409       15125
turno 8                        1330            3524                3524       18649
turno 9             

**La columna «según el servidor» tiene que coincidir con la propia**, salvo un par de tokens de la plantilla. Mientras coincidan, nada se cortó. La sección 4 rompe esa igualdad a propósito.

### 3.2 · Y el buscador, con la conversación entera

La razón que quedó pendiente en la sección 2. Si en cada turno buscáramos con la conversación entera concatenada en orden, ¿cuántos tokens del embedder serían? El pipeline corta a 1.024, por la derecha, sin avisar.

In [32]:
TOPE_EMB = embedder.max_seq_length
corte, filas = None, []
for t in TURNOS[1:]:
    previo = HIST[:2 * t["i"]]                       # la conversacion antes de este turno
    n_previo = contar_emb(texto_de(previo))
    n_total = contar_emb(texto_de(previo + [{"role": "user", "content": t["turno"]}]))
    visible = "entero" if n_total <= TOPE_EMB else (
        "a medias" if n_previo < TOPE_EMB else "AFUERA")
    filas.append(("turno %d" % t["i"], {"tokens de la consulta": n_total,
                                        "el turno nuevo arranca en": n_previo,
                                        "¿el embedder lo ve?": visible}))
    if visible == "AFUERA" and corte is None:
        corte = t["i"]
tabla("La conversación entera como consulta · tope del embedder: %d tokens" % TOPE_EMB,
      ["tokens de la consulta", "el turno nuevo arranca en", "¿el embedder lo ve?"], filas)

if corte is not None:
    previo = HIST[:2 * corte]
    consulta = texto_de(previo + [{"role": "user", "content": GUION[corte]["turno"]}])
    ids = TOK_EMB.encode(consulta, add_special_tokens=False)[:TOPE_EMB - 2]
    print("En el turno %d, lo último que el embedder llega a leer es:" % corte)
    print(parrafo("…" + TOK_EMB.decode(ids)[-220:]))
    print("y el turno nuevo, «%s», queda entero del otro lado del corte." % GUION[corte]["turno"])
    print()
    # Los resultados repetidos: tres turnos seguidos despues del corte.
    for i in range(corte, min(corte + 3, len(GUION))):
        c = texto_de(HIST[:2 * i] + [{"role": "user", "content": GUION[i]["turno"]}])
        print("  turno %d · conversación entera → %s" % (i, " ".join(a_documentos(buscar(c)))))
    print()
    print("El canal denso ve siempre los mismos primeros 1.024 tokens, así que trae lo mismo;")
    print("BM25 no tiene tope y lee todo. Los dos canales terminan buscando textos distintos.")
    # El parche: el turno nuevo primero.
    invertida = texto_de([{"role": "user", "content": GUION[corte]["turno"]}] + previo[::-1])
    print("\nParche: con el turno nuevo PRIMERO, arranca en el token 0 y entra siempre.")
    print("  turno %d · turno nuevo primero → %s"
          % (corte, " ".join(a_documentos(buscar(invertida)))))
    print("La otra razón —un vector que promedia toda la conversación— no tiene parche.")
else:
    print("La conversación no llegó a pasar el tope del embedder.")
RESULTADOS["3 · corte del embedder"] = {"turno del corte": corte, "tope": TOPE_EMB}


La conversación entera como consulta · tope del embedder: 1024 tokens
──────────────────────────────────────────────────────────────────────────────────────
               tokens de la consulta   el turno nuevo arranca en   ¿el embedder lo ve?
turno 1                           92                          71                entero
turno 2                          168                         155                entero
turno 3                          285                         266                entero
turno 4                          392                         384                entero
turno 5                          562                         535                entero
turno 6                          671                         660                entero
turno 7                          869                         839                entero
turno 8                          936                         902                entero
turno 9                         1121                       

### 3.3 · Leer y escribir son dos trabajos distintos

Generar tiene dos fases. El **prefill** lee el prompt entero de una vez, en paralelo; el **decode** escribe la respuesta de a un token. Para separarlas, re-enviamos la misma conversación turno por turno —sólo la conversación, sin pasajes— y pedimos 32 tokens de respuesta. Con un detalle: el mensaje de sistema lleva **la hora, con microsegundos**. Es lo que hacen muchos productos para que el modelo sepa qué día es, y tiene una consecuencia que la sección 3.5 mide: cada llamada empieza distinta, así que el servidor no puede reusar nada de la anterior. Acá eso es lo que queremos: un prefill completo en cada turno.

In [33]:
def repetir(sistema, n_predict=32):
    """La conversacion re-enviada turno por turno, sin pasajes. sistema() arma el mensaje de
    sistema en cada llamada: si cambia, el cache de prefijos no sirve."""
    filas = []
    for n in range(1, len(GUION)):
        msgs = [{"role": "system", "content": sistema()}] + HIST[:2 * n] + \
               [{"role": "user", "content": GUION[n]["turno"]}]
        r = chat(msgs, num_predict=n_predict)
        filas.append({"n": n, "prompt": r["prompt"], "prefill": r["prefill_s"],
                      "decode": r["decode_s"], "salida": r["salida"]})
    return filas

con_hora = lambda: "Fecha y hora: %s.\n%s" % (
    datetime.datetime.now().isoformat(timespec="microseconds"), SISTEMA)

chat([{"role": "user", "content": "ok"}], num_predict=1)     # el modelo cargado con NUM_CTX
FRIO = repetir(con_hora)
tabla("Prefill completo en cada turno",
      ["tokens", "prefill (s)", "decode (ms por token)"],
      [("turno %d" % f["n"], {"tokens": f["prompt"], "prefill (s)": "%.2f" % f["prefill"],
                              "decode (ms por token)": "%.0f" % (1000 * f["decode"] / max(1, f["salida"]))})
       for f in FRIO])
print("El prefill crece con la conversación; el costo de cada token de salida, no.")
print("El historial se paga antes de la primera palabra: eso es el tiempo hasta el primer token.")
RESULTADOS["3 · prefill frío"] = {"turno 1": round(FRIO[0]["prefill"], 2),
                                  "turno %d" % FRIO[-1]["n"]: round(FRIO[-1]["prefill"], 2)}


Prefill completo en cada turno
───────────────────────────────────────────────────────────
               tokens   prefill (s)   decode (ms por token)
turno 1           275          0.32                      33
turno 2           400          0.36                      32
turno 3           562          0.50                      33
turno 4           728          0.69                      33
turno 5           978          0.94                      34
turno 6          1134          1.05                      33
turno 7          1441          1.38                      34
turno 8          1523          1.44                      35
turno 9          1785          1.69                      35
turno 10         2022          1.94                      34
turno 11         2244          2.14                      34
turno 12         2524          2.44                      35
turno 13         2638          2.54                      35
turno 14         2887          2.81                      35
turno 15

### 3.4 · El KV cache: lo que guarda mientras escribe

Para escribir cada token, la atención necesita las claves y los valores de todos los tokens anteriores, en cada capa. Esos no cambian, así que se guardan: es el **cache de claves y valores** (*key-value cache*, o KV cache). Y tiene un tamaño que se puede calcular desde la configuración del modelo, que se baja sin bajar el modelo: por cada token, una clave y un valor, en cada capa, para cada cabeza de clave y valor, con la dimensión de la cabeza, en bytes de 16 bits.

In [34]:
# La cuenta, desde la configuracion del modelo: se baja sin bajar el modelo.
CFG = json.load(open(hf_hub_download("Qwen/Qwen3-8B", "config.json")))

capas, h = CFG["num_hidden_layers"], CFG["hidden_size"]
cab_q, cab_kv = CFG["num_attention_heads"], CFG["num_key_value_heads"]
dim = CFG.get("head_dim") or h // cab_q

KV_POR_TOKEN = 2 * capas * cab_kv * dim * 2           # clave y valor · capas · cabezas · dim · 2 bytes

print("configuración: %d capas · %d cabezas de consulta · %d de clave y valor · dimensión %d"
      % (capas, cab_q, cab_kv, dim))
print("KV cache por token: 2 × %d × %d × %d × 2 B = %s B = %.0f KiB   (un KiB son 1.024 bytes)"
      % (capas, cab_kv, dim, "{:,}".format(KV_POR_TOKEN).replace(",", "."), KV_POR_TOKEN / 1024))
for n in (4096, 32768, 131072):
    print("  una ventana de %7s tokens ocupa %5.2f GiB" % ("{:,}".format(n).replace(",", "."),
                                                        KV_POR_TOKEN * n / 2**30))

configuración: 36 capas · 32 cabezas de consulta · 8 de clave y valor · dimensión 128
KV cache por token: 2 × 36 × 8 × 128 × 2 B = 147.456 B = 144 KiB   (un KiB son 1.024 bytes)
  una ventana de   4.096 tokens ocupa  0.56 GiB
  una ventana de  32.768 tokens ocupa  4.50 GiB
  una ventana de 131.072 tokens ocupa 18.00 GiB


**La ventana es, físicamente, este cache**, y el servidor la **reserva** al cargar el modelo. Para comprobar la cuenta, cargamos el generador dos veces, con ventanas de 4.096 y de 32.768 tokens, y le preguntamos al servidor cuánta memoria toma cada vez. La diferencia tiene que parecerse a la cuenta: 144 KiB por cada token de ventana de más.

In [35]:
def medir_ventana(num_ctx):
    liberar_gpu(MODEL)                                   # descargarlo: que cargue de cero
    time.sleep(2)
    chat([{"role": "user", "content": "ok"}], num_ctx=num_ctx, num_predict=1)
    return modelo_en_placa()

liberar_gpu()
MEDIDAS_KV = {n: medir_ventana(n) for n in (4096, 32768)}
donde_corre()

filas = []
for n, m in MEDIDAS_KV.items():
    if m:
        filas.append(("ventana %d" % n, {"memoria total": "%.2f GiB" % (m["total"] / 2**30),
                                         "en la GPU": "%.2f GiB" % (m["gpu"] / 2**30),
                                         "ventana informada": m["ventana"] or "—"}))
if len(filas) == 2:
    tabla("El mismo modelo, dos reservas de ventana", ["memoria total", "en la GPU", "ventana informada"], filas)
    medida = (MEDIDAS_KV[32768]["total"] - MEDIDAS_KV[4096]["total"]) / 2**30
    calculada = KV_POR_TOKEN * (32768 - 4096) / 2**30
    print("diferencia medida   : %.2f GiB" % medida)
    print("diferencia calculada: %.2f GiB  (144 KiB × 28.672 tokens de ventana de más)" % calculada)
    print("Lo que no coincide es la memoria de trabajo del servidor, que también crece con la ventana.")
    RESULTADOS["3 · KV cache"] = {"4K (GiB)": round(MEDIDAS_KV[4096]["total"] / 2**30, 2),
                                  "32K (GiB)": round(MEDIDAS_KV[32768]["total"] / 2**30, 2),
                                  "medida": round(medida, 2), "calculada": round(calculada, 2)}
else:
    print("No pude leer la memoria del modelo cargado (¿Ollama está vivo?).")

# De vuelta a la ventana de trabajo del notebook.
liberar_gpu(MODEL)
chat([{"role": "user", "content": "ok"}], num_predict=1)

NAME        ID              SIZE      PROCESSOR    CONTEXT    RUNNER      UNTIL              
qwen3:8b    500a1f067a9f    9.8 GB    100% GPU     32768      llamacpp    4 minutes from now

El mismo modelo, dos reservas de ventana
───────────────────────────────────────────────────────────────
                  memoria total   en la GPU   ventana informada
ventana 4096           5.20 GiB    5.20 GiB                4096
ventana 32768          9.16 GiB    9.16 GiB               32768

diferencia medida   : 3.96 GiB
diferencia calculada: 3.94 GiB  (144 KiB × 28.672 tokens de ventana de más)
Lo que no coincide es la memoria de trabajo del servidor, que también crece con la ventana.


{'texto': 'Hello',
 'mensaje': {'role': 'assistant', 'content': 'Hello'},
 'prompt': 17,
 'prefill_s': 0.132259999,
 'salida': 1,
 'decode_s': 1e-06,
 'error': None}

#### Para profundizar · Las dos memorias, en bits

Las dos memorias de la sección 1, ahora con números. **Los pesos:** aun si cada bit guardara texto —que no lo hace—, ¿a cuántos bits por token de entrenamiento tocaría? Los parámetros se cuentan desde la configuración; los 36 billones de tokens de entrenamiento son la cifra que publica el fabricante para la familia. **La ventana:** lo que ocupa un token mientras está en el KV cache, la cuenta de arriba.

In [36]:
# Los parametros, contados desde la configuracion: atencion + MLP por capa, mas los embeddings.
atencion = h * (cab_q * dim) + 2 * h * (cab_kv * dim) + (cab_q * dim) * h
mlp = 3 * h * CFG["intermediate_size"]
emb = CFG["vocab_size"] * h * (1 if CFG.get("tie_word_embeddings") else 2)
PARAMETROS = capas * (atencion + mlp) + emb

TOKENS_ENTRENAMIENTO = 36e12                           # blog de Qwen3: ~36 billones (la familia)
bits_por_token = PARAMETROS * 16 / TOKENS_ENTRENAMIENTO
bits_tal_cual = np.log2(CFG["vocab_size"])             # anotar un token sin comprimir

print("LOS PESOS")
print("  parámetros (contados desde la config) : %.2f mil millones" % (PARAMETROS / 1e9))
print("  le tocan a cada token de entrenamiento: %.4f bits (cota: como si cada bit guardara texto)"
      % bits_por_token)
print("  anotar un token tal cual pide         : %.1f bits (log2 de un vocabulario de %s)"
      % (bits_tal_cual, "{:,}".format(CFG["vocab_size"]).replace(",", ".")))
print("LA VENTANA")
print("  un token en el KV cache ocupa         : %s bits"
      % "{:,}".format(KV_POR_TOKEN * 8).replace(",", "."))
print()
print("Anotar un token pide %.0f veces lo que le toca en los pesos; en la ventana ocupa %.1e veces."
      % (bits_tal_cual / bits_por_token, KV_POR_TOKEN * 8 / bits_por_token))
RESULTADOS["3 · dos memorias, en bits"] = {"KV por token (B)": KV_POR_TOKEN,
                                           "parámetros": PARAMETROS,
                                           "bits por token de entrenamiento": round(bits_por_token, 4)}

LOS PESOS
  parámetros (contados desde la config) : 8.19 mil millones
  le tocan a cada token de entrenamiento: 0.0036 bits (cota: como si cada bit guardara texto)
  anotar un token tal cual pide         : 17.2 bits (log2 de un vocabulario de 151.936)
LA VENTANA
  un token en el KV cache ocupa         : 1.179.648 bits

Anotar un token pide 4729 veces lo que le toca en los pesos; en la ventana ocupa 3.2e+08 veces.


#### Para profundizar · Por qué 8 cabezas de clave y valor y no 32

El modelo tiene 32 cabezas de consulta, pero las agrupa de a cuatro para compartir una clave y un valor: es la **atención de consultas agrupadas** (*grouped-query attention*, GQA). Sin eso, el cache sería cuatro veces más grande; con un solo par para todas (*multi-query*), mucho más chico, con más pérdida de calidad. El tamaño de la memoria de trabajo es una decisión de arquitectura.

In [37]:
print("KV cache por token, según cuántos pares de clave y valor tenga cada capa:")
print("  %2d pares, uno por cabeza de consulta (multi-head) : %4.0f KiB"
      % (cab_q, 2 * capas * cab_q * dim * 2 / 1024))
print("  %2d pares, uno cada %d cabezas (GQA, este modelo)  : %4.0f KiB"
      % (cab_kv, cab_q // cab_kv, KV_POR_TOKEN / 1024))
print("  %2d par para todas las cabezas (multi-query)      : %4.0f KiB"
      % (1, 2 * capas * 1 * dim * 2 / 1024))

KV cache por token, según cuántos pares de clave y valor tenga cada capa:
  32 pares, uno por cabeza de consulta (multi-head) :  576 KiB
   8 pares, uno cada 4 cabezas (GQA, este modelo)  :  144 KiB
   1 par para todas las cabezas (multi-query)      :   18 KiB


### 3.5 · No volver a leer lo que ya se leyó

La misma repetición de la sección 3.3, ahora con el mensaje de sistema **fijo**. Entre un turno y el siguiente, casi todo el prompt se repite: si el servidor guardó el cache de la llamada anterior, reusa el de la parte que coincide y hace prefill sólo de lo nuevo.

*Nota de laboratorio:* Ollama reusa el cache del prefijo mientras el modelo siga cargado; su conteo de tokens del prompt incluye los que vinieron del cache, así que el ahorro se ve en el tiempo, no en el conteo.

In [38]:
ESTABLE = repetir(lambda: SISTEMA)

filas = [("turno %d" % f["n"], {"tokens": f["prompt"], "con la hora arriba (s)": "%.2f" % f["prefill"],
                                "con prefijo estable (s)": "%.2f" % e["prefill"]})
         for f, e in zip(FRIO, ESTABLE)]
tabla("Tiempo de prefill por turno, con y sin prefijo estable",
      ["tokens", "con la hora arriba (s)", "con prefijo estable (s)"], filas)
tf, te = sum(f["prefill"] for f in FRIO), sum(e["prefill"] for e in ESTABLE)
print("prefill de toda la conversación · con la hora arriba: %.1f s · con prefijo estable: %.1f s"
      % (tf, te))
print("Cambiar una línea arriba invalida todo lo de abajo.")
RESULTADOS["3 · cache de prefijos"] = {"prefill total frío (s)": round(tf, 1),
                                      "prefill total estable (s)": round(te, 1)}


Tiempo de prefill por turno, con y sin prefijo estable
────────────────────────────────────────────────────────────────────────
               tokens   con la hora arriba (s)   con prefijo estable (s)
turno 1           275                     0.32                      0.24
turno 2           400                     0.36                      0.18
turno 3           562                     0.50                      0.20
turno 4           728                     0.69                      0.20
turno 5           978                     0.94                      0.30
turno 6          1134                     1.05                      0.21
turno 7          1441                     1.38                      0.35
turno 8          1523                     1.44                      0.12
turno 9          1785                     1.69                      0.31
turno 10         2022                     1.94                      0.27
turno 11         2244                     2.14                      

**Un aviso para lo que viene.** Recortar o resumir el historial reescribe el prefijo, y rompe el cache igual que la hora: cada compactación cuesta una relectura completa.

▶ **Láminas 18–23** (el turno 20 manda veinte turnos; prefill y decode; el KV cache; las dos memorias, en bits; GQA; el prefijo estable). Y la **lámina 13**: dónde corta el embedder.

## 4 · La ventana se acaba

*Momento 2 · La conversación larga (sigue)*

El cache tiene techo: la ventana. Hasta acá le pedimos 8.192 tokens en cada llamada, y la conversación entraba. ¿Qué pasa cuando no entra? Primero, un dato: ese techo no es uno solo. El modelo tiene el máximo que aprendió en el entrenamiento, y quien lo sirve decide cuánta ventana **reservar** por pedido. ¿Qué asigna el servidor si nadie le pide nada?

In [39]:
def ventana_del_servidor():
    """La ventana con la que quedo cargado el modelo. Si la API no la informa, se lee de 'ollama ps'."""
    m = modelo_en_placa()
    if m and m.get("ventana"):
        return int(m["ventana"])
    salida = subprocess.run(["ollama", "ps"], capture_output=True, text=True).stdout.splitlines()
    if len(salida) >= 2 and "CONTEXT" in salida[0]:
        cols = re.split(r"\s{2,}", salida[0].strip())
        for linea in salida[1:]:
            vals = re.split(r"\s{2,}", linea.strip())
            if vals and vals[0].startswith(MODEL) and len(vals) == len(cols):
                return int(vals[cols.index("CONTEXT")])
    return None

liberar_gpu(MODEL)
chat([{"role": "user", "content": "ok"}], num_ctx=None, num_predict=1)   # SIN pedir ventana
VENTANA_SERVIDOR = ventana_del_servidor() or 4096
print("máximo entrenado del modelo (config)       :", CFG.get("max_position_embeddings"))
print("ventana asignada por el servidor, por default:", VENTANA_SERVIDOR)
print("el prompt del turno %d de nuestra conversación: %d tokens" % (COSTO[-1]["i"], COSTO[-1]["total"]))

máximo entrenado del modelo (config)       : 40960
ventana asignada por el servidor, por default: 4096
el prompt del turno 20 de nuestra conversación: 6148 tokens


*Nota de laboratorio:* Ollama asigna por defecto 4.096 tokens en GPUs de menos de 24 GiB, como una T4, y su endpoint compatible con OpenAI no deja cambiarla por pedido. La documentación vieja dice 2.048: se mide, no se cree.

### 4.1 · Cuando no entra, algo se cae

Provocado a propósito. Después del turno 20, el usuario pregunta algo que se dijo en el turno 0. La pregunta pasa por el pipeline de siempre —historial entero más pasajes—, pero **sin pedir ventana**: con la que asigna el servidor.

In [40]:
# La pregunta apunta a la conversacion ("te dije"): con pasajes en el turno, una pregunta neutra
# lleva al modelo a buscar la respuesta en los pasajes y abstenerse, entre o no entre el historial.
PREGUNTA_CIUDAD = "¿En qué ciudad te dije que trabajo?"

# El turno de siempre: historial entero + pasajes. Si la conversacion de esta corrida salio
# corta y todavia entra en la ventana del servidor, pedimos mas pasajes hasta que no entre:
# el desborde se provoca a proposito, y la celda dice con cuantos.
k_ciudad = K
while True:
    pas_ciudad = buscar(PREGUNTA_CIUDAD, k_ciudad)
    msgs_ciudad = [{"role": "system", "content": SISTEMA}] + HIST + \
                  [{"role": "user", "content": mensaje_usuario(PREGUNTA_CIUDAD, pas_ciudad)}]
    propia = contar_prompt(msgs_ciudad)
    if propia > VENTANA_SERVIDOR + 200 or k_ciudad >= 20:
        break
    k_ciudad += 2
print("pasajes en el turno: %d%s" % (k_ciudad, "" if k_ciudad == K else
      "  (más que los %d de siempre, para que no entre)" % K))
obj, r = estructurado(msgs_ciudad, RespuestaChat, num_predict=600, num_ctx=None)
respuesta = solo_respuesta(obj, r)

print("ventana del servidor        :", VENTANA_SERVIDOR)
print("tokens del prompt · propios :", propia)
print("tokens del prompt · servidor:", r["prompt"])
print("respuesta                   :", respuesta)
print("¿dice Vigo?                 :", "sí" if "vigo" in respuesta.lower() else "NO")
if propia <= VENTANA_SERVIDOR:
    print("\n⚠ la conversación entró en la ventana: esta corrida no desbordó.")
RESULTADOS["4 · desborde silencioso"] = {"ventana": VENTANA_SERVIDOR, "propios": propia,
                                         "servidor": r["prompt"], "respuesta": respuesta}

pasajes en el turno: 5
ventana del servidor        : 4096
tokens del prompt · propios : 6170
tokens del prompt · servidor: 4081
respuesta                   : No se menciona ninguna ciudad específica en el contexto proporcionado.
¿dice Vigo?                 : NO


**Lo que pasó.** El servidor no rechazó nada: tiró los mensajes más viejos hasta que el resto entrara —conservando el de sistema— y contestó con fluidez. El turno 0 no llegó al modelo. Desde afuera, una respuesta que olvidó el turno 0 parece un modelo que «se olvidó», y es la infraestructura.

*Nota de laboratorio:* Ollama usa las políticas silenciosas: en un chat tira los mensajes más viejos y conserva el de sistema; un prompt único que no entra pierde el medio; y si falta lugar para la respuesta, corre la ventana a mitad de camino. Todo con una respuesta normal.

### 4.2 · Contar antes de mandar

La defensa es aburrida y funciona: contar con el tokenizer del propio modelo, sumar el lugar para la respuesta y comparar con la ventana. Y, si se puede, pedirle al servidor que **rechace** en vez de recortar: con `truncate: false` y `shift: false`, Ollama convierte los recortes silenciosos en errores.

In [41]:
RESERVA = 600     # lugar para la respuesta

def cabe(mensajes, ventana, reserva=RESERVA):
    n = contar_prompt(mensajes)
    return n + reserva <= ventana, n

entra, n = cabe(msgs_ciudad, VENTANA_SERVIDOR)
print("¿prompt + respuesta ≤ ventana?  %d + %d ≤ %d → %s" % (n, RESERVA, VENTANA_SERVIDOR,
                                                           "sí" if entra else "NO: hay que decidir qué sacar"))
print()
r_err = chat(msgs_ciudad, num_ctx=None, num_predict=300, extra={"truncate": False, "shift": False})
print("el mismo pedido, con truncate=false y shift=false:")
print(parrafo(r_err["error"] or "(el servidor no lo rechazó: %s)" % recortar(r_err["texto"], 120)))
print()
obj, r = estructurado(msgs_ciudad, RespuestaChat, num_predict=600, num_ctx=NUM_CTX)
print("con la ventana que entra (%d) · servidor %d tokens · respuesta: %s"
      % (NUM_CTX, r["prompt"], solo_respuesta(obj, r)))
RESULTADOS["4 · contar antes de mandar"] = {"error explícito": (r_err["error"] or "")[:160]}

¿prompt + respuesta ≤ ventana?  6170 + 600 ≤ 4096 → NO: hay que decidir qué sacar

el mismo pedido, con truncate=false y shift=false:
   HTTP 400 · {"error":"{\"error\":{\"code\":400,\"message\":\"request (6170
   tokens) exceeds the available context size (4096 tokens), try increasing i
   t\",\"type\":\"exceed_context_size_error\",\"n_prompt_tokens\":6170,\"n_ct
   x\":4096}}"}

con la ventana que entra (8192) · servidor 6170 tokens · respuesta: Trabajas en Vigo.


**Si el harness no cuenta, recorta el servidor, y recorta lo que no debía.** Contar convierte una falla silenciosa en una decisión: si el harness sabe que no entra, decide él qué se cae. Ésa es la sección 5.

Dos láminas de esta sección no tienen celda, porque sus números son publicados y no se reproducen en una T4: la **ventana nominal contra la efectiva** (RULER, NoLiMa y la curva del propio fabricante del modelo) y **por qué hay un techo**.

▶ **Láminas 24–26** (cuando no entra, algo se cae; dos límites; contar antes de mandar). Sin celda: 27 y 28.

## 5 · Repartir la ventana

*Momento 2 · La conversación larga (sigue)*

Si la ventana es un presupuesto, primero hay que saber quién lo gasta. En un turno de RAG conversacional son seis: las **instrucciones**, la **memoria** de sesiones anteriores (vacía hasta la semana que viene, desde la sección 9), el **historial**, los **pasajes**, la **pregunta** y la **reserva para la respuesta**. Sólo uno crece solo: el historial.

El harness deja de pasar una lista que crece y **arma la ventana en cada turno**: primero lo fijo, después los pasajes del mejor al peor hasta su tope, después el historial del turno más nuevo al más viejo hasta lo que quede. Contar, y mandar. Usamos de presupuesto la ventana que asigna el servidor.

In [42]:
PRESUPUESTO  = VENTANA_SERVIDOR    # la ventana de 4K que asigna el servidor, ahora como presupuesto
TOPE_PASAJES = 1600

def costo_par(par):
    """Tokens de un par usuario-asistente en la plantilla (el contenido y sus marcas de turno)."""
    return sum(contar(m["content"]) + 5 for m in par)

def ensamblar(pregunta, historial, pasajes, memoria="", presupuesto=None, reserva=RESERVA,
              tope_pasajes=TOPE_PASAJES, tope_historial=None, anclas=0, sistema=SISTEMA):
    """Arma la ventana de un turno adentro de un presupuesto contado.

    Orden de llenado: lo fijo (instrucciones, memoria, pregunta, reserva) -> los pasajes del
    mejor al peor hasta su tope -> el historial, del turno mas nuevo al mas viejo, hasta lo que
    quede (o hasta tope_historial, si se fija: es el reparto entre pasajes e historial).
    anclas: cuantos pares del principio del historial entran SIEMPRE (seccion 6).

    Devuelve (mensajes, cuentas por inquilino, pares que quedaron afuera). Lo que queda afuera
    lo decide el harness, a la vista: no el servidor, en silencio."""
    presupuesto = presupuesto or PRESUPUESTO
    sis = sistema + ("\n\n<memoria>\n%s\n</memoria>" % memoria if memoria else "")
    elegidos, usados = [], 0
    for p in pasajes:
        c = contar(armar_contexto([p])) + 2
        if usados + c > tope_pasajes:
            break
        elegidos.append(p)
        usados += c
    base = [{"role": "system", "content": sis}]
    ultimo = {"role": "user", "content": mensaje_usuario(pregunta, elegidos)}
    resto = presupuesto - reserva - contar_prompt(base + [ultimo])
    if tope_historial is not None:
        resto = min(resto, tope_historial)
    pares = [historial[j:j + 2] for j in range(0, len(historial), 2)]
    fijos, moviles = pares[:anclas], pares[anclas:]
    resto -= sum(costo_par(p) for p in fijos)
    quedan = []
    for par in reversed(moviles):                 # del turno mas nuevo al mas viejo
        c = costo_par(par)
        if c > resto:
            break                                 # el primero que no entra corta: sin huecos
        quedan.insert(0, par)
        resto -= c
    afuera = moviles[:len(moviles) - len(quedan)]
    msgs = base + [m for par in fijos + quedan for m in par] + [ultimo]
    cuentas = {"instrucciones": contar(sistema), "memoria": contar(memoria) if memoria else 0,
               "historial": sum(costo_par(p) for p in fijos + quedan),
               "pasajes": contar(armar_contexto(elegidos)) if elegidos else 0,
               "pregunta": contar(pregunta), "reserva": reserva}
    cuentas["total"] = contar_prompt(msgs) + reserva
    cuentas["turnos afuera"] = len(afuera)
    return msgs, cuentas, afuera

INQUILINOS = ["instrucciones", "memoria", "historial", "pasajes", "pregunta", "reserva",
              "total", "turnos afuera"]
filas = []
for i in (2, 10, len(GUION) - 1):
    if i < len(TURNOS):
        _, cuentas, _ = ensamblar(GUION[i]["turno"], HIST[:2 * i], TURNOS[i]["salida"]["pasajes"])
        filas.append(("turno %d" % i, cuentas))
tabla("Los seis inquilinos, en tokens · presupuesto %d" % PRESUPUESTO, INQUILINOS, filas)
RESULTADOS["5 · inquilinos"] = dict(filas)


Los seis inquilinos, en tokens · presupuesto 4096
─────────────────────────────────────────────────────────────────────────────────────────────────────────
               instrucciones   memoria   historial   pasajes   pregunta   reserva   total   turnos afuera
turno 2                  107         0         230      1574         10       600    2552               0
turno 10                 107         0        1748      1568         26       600    4081               1
turno 20                 107         0        1707      1588         33       600    4067              11



**Cómo leerlo.** En el turno 2 sobra lugar. Mirá la columna de turnos afuera: cuando el historial ya no entra entero, el ensamblador **decide** qué se cae —los turnos más viejos, a la vista y contados—. Es la misma pérdida que la sección 4 sufrió en silencio, ahora como decisión.

La prueba de que no hay sorpresas: mandamos la ventana del turno 20 con la ventana del servidor, y la cuenta propia tiene que coincidir con la del servidor.

In [43]:
ULT = len(GUION) - 1                                     # el turno 20
msgs20, cuentas20, afuera20 = ensamblar(GUION[ULT]["turno"], HIST[:2 * ULT], TURNOS[ULT]["salida"]["pasajes"])
obj, r = estructurado(msgs20, RespuestaChat, num_predict=RESERVA, num_ctx=PRESUPUESTO)
print("ventana armada para el turno %d · propia: %d tokens · según el servidor: %d"
      % (ULT, contar_prompt(msgs20), r["prompt"]))
AFUERA20 = list(range(len(afuera20)))                    # salen los mas viejos, sin huecos
print("turnos que el ensamblador dejó afuera (%d):" % len(AFUERA20),
      ", ".join(str(j) for j in AFUERA20) or "ninguno")
for j, dato in ((0, "Vigo, y las respuestas con el artículo"), (7, "el 1256/2011, artículo 9")):
    print("   el turno %d · %-38s → %s" % (j, dato, "AFUERA" if j in AFUERA20 else "adentro"))
print("respuesta:", recortar(solo_respuesta(obj, r), 200))
RESULTADOS["5 · la ventana del turno 20"] = {"turnos afuera": AFUERA20}

ventana armada para el turno 20 · propia: 3467 tokens · según el servidor: 3467
turnos que el ensamblador dejó afuera (11): 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10
   el turno 0 · Vigo, y las respuestas con el artículo → AFUERA
   el turno 7 · el 1256/2011, artículo 9               → AFUERA
respuesta: (JSON cortado; su análisis:) El programa «Justicia» fue el programa de financiación que sostuvo entre 2014 y 2020 las acciones para construir un espacio común de justicia, según el Reglamento (UE) No …


**Desde acá, cada técnica es una regla nueva del ensamblador**: qué entra a cada segmento y en qué forma.

### 5.1 · El historial guarda lo que se dijo

Al historial va lo que vio el usuario: la respuesta y su cita. No el análisis del schema, ni los pasajes —se recuperan otra vez en cada turno—, ni el razonamiento del modelo. Abajo, lo que produjo un turno y lo que quedó guardado.

In [44]:
t = TURNOS[2]["salida"]
print("LO QUE PRODUJO EL TURNO 2")
print("  análisis   :", recortar(t["analisis"], 100), "  ← no va al historial")
print("  pasajes    : %d, %d tokens" % (len(t["pasajes"]), contar(armar_contexto(t["pasajes"]))),
      "  ← no va: se recuperan otra vez")
print("  respuesta  :", recortar(t["respuesta"], 100))
print("  cita       :", " ".join(t["cita"]))
print()
print("LO QUE QUEDÓ EN EL HISTORIAL")
print(parrafo(HIST[5]["content"]))

LO QUE PRODUJO EL TURNO 2
  análisis   : El Reglamento (UE) No 1256/2011 establece que las posibilidades de pesca para las poblaciones de bac…   ← no va al historial
  pasajes    : 5, 1574 tokens   ← no va: se recuperan otra vez
  respuesta  : El Reglamento (UE) No 1256/2011 establece que las posibilidades de pesca para las poblaciones de bac…
  cita       : [32011R1256 · encabezado] [32011R1256 · encabezado]

LO QUE QUEDÓ EN EL HISTORIAL
   El Reglamento (UE) No 1256/2011 establece que las posibilidades de pesca
   para las poblaciones de bacalao del Mar Báltico deben seguir las normas
   del Reglamento (CE) no 1098/2007, que es el plan plurianual para estas
   poblaciones. Además, se introduce flexibilidad en la gestión del esfuerzo
   pesquero sin comprometer los objetivos del plan. [32011R1256 · encabezado]
   [32011R1256 · encabezado]


*Nota de laboratorio:* el fabricante del modelo recomienda no guardar el razonamiento en el historial, y su plantilla de chat ya lo saca. Se ve renderizando un turno viejo que traía razonamiento adentro:

In [45]:
con_razonamiento = [
    {"role": "user", "content": "¿Y qué dice sobre el bacalao?"},
    {"role": "assistant", "content": "<think>El usuario probablemente se refiere al Báltico de "
                                     "2012.</think>Se rige por el plan plurianual del bacalao."},
    {"role": "user", "content": "¿Desde cuándo?"}]
texto = plantilla(con_razonamiento)
print(texto)
print("¿el razonamiento del turno viejo llegó al prompt?", "sí" if "probablemente" in texto else "no")

<|im_start|>user
¿Y qué dice sobre el bacalao?<|im_end|>
<|im_start|>assistant
Se rige por el plan plurianual del bacalao.<|im_end|>
<|im_start|>user
¿Desde cuándo?<|im_end|>
<|im_start|>assistant
<think>

</think>


¿el razonamiento del turno viejo llegó al prompt? no


**Si entra, se pega.** El criterio antes de cualquier técnica de memoria: si el historial entero entra en el presupuesto, y el costo y la calidad aguantan, re-enviarlo es la vara a superar. Acá, con la ventana de 8.192 que pedimos en la sección 3, el turno 20 todavía entraba; con la de 4.096, ya no. La sección 6 es lo que se hace cuando no entra.

> **Estamos acá · fin del momento 2, la conversación larga.** **Resuelto:** sabemos cuánto cuesta re-enviar todo y cómo abaratarlo —un principio del prompt estable, que el servidor no vuelve a leer—, sabemos dónde se acaba la ventana y que, si nadie cuenta, el servidor recorta en silencio, y el harness arma la ventana en cada turno, adentro de un presupuesto contado. **Costó:** contar en cada turno y decidir un reparto entre pasajes e historial. **El problema nuevo:** con la ventana de 4.096, en el turno 20 el ensamblador ya deja turnos afuera, y la ventana del turno 20, más arriba, dice si entre ellos están el 0 (Vigo) y el 7 (el artículo 9). Ahora lo decide el harness, a la vista; pero tirar los más viejos es sólo una forma de sacar algo, y no la mejor.

▶ **Láminas 29–34** (los inquilinos; el reparto; el ensamblador; qué va al historial; si entra, se pega; estamos acá).

## 6 · Qué sacar de la ventana

*Momento 3 · Cuando ya no entra*

El historial pasó su presupuesto y algo hay que sacar. Toda política de olvido es **una apuesta sobre qué va a importar**, hecha antes de que exista la pregunta que lo va a necesitar. **Recortar** apuesta a lo reciente; **anclar** emparcha lo primero; **resumir** apuesta a lo que el resumidor cree importante; **descargar** no apuesta: saca de la ventana sin destruir y deja un puntero.

El experimento es uno solo. Después del turno 20, el usuario pregunta por algo que dijo en el turno 0: **en qué ciudad trabaja**. La pregunta es sobre la conversación, no sobre el corpus, así que no se busca: la ventana se la reparten las instrucciones y el historial. Al historial le damos un presupuesto fijo —lo que ocupan los últimos ocho turnos: el reparto de la sección 5— y cada política lo llena a su manera. El control es pegar todo con la ventana de 8.192.

In [46]:
ULTIMOS = 8                    # la ventana deslizante: los ultimos ocho turnos
PALABRAS_RESUMEN = 100         # el resumen tambien tiene presupuesto
PRUEBA_CIUDAD  = "¿En qué ciudad trabajo?"
PRUEBA_INFORME = "¿Qué reglamento y qué artículo te dije que iba a citar en el informe?"

# El presupuesto del historial es lo que ocupan los ultimos ocho turnos. Asi las cuatro
# politicas pierden exactamente lo mismo -los turnos del principio- y lo que cambia es que
# hace cada una con eso.
PARES = [HIST[j:j + 2] for j in range(0, len(HIST), 2)]
TOPE_HISTORIAL = sum(costo_par(p) for p in PARES[-ULTIMOS:]) + 20
N_AFUERA = max(0, len(PARES) - ULTIMOS)       # los turnos 0 .. N_AFUERA-1 no entran textuales
RECIENTES = HIST[2 * N_AFUERA:]
print("presupuesto del historial: %d tokens, lo que ocupan los últimos %d turnos" % (TOPE_HISTORIAL, ULTIMOS))
print("entran textuales los turnos %d a %d; quedan afuera del 0 al %d"
      % (N_AFUERA, len(PARES) - 1, N_AFUERA - 1))

presupuesto del historial: 1648 tokens, lo que ocupan los últimos 8 turnos
entran textuales los turnos 13 a 20; quedan afuera del 0 al 12


### 6.1 · Resumir: un resumen que se reescribe

Los turnos que salen se condensan en un **resumen corrido** (*running summary*): resumen nuevo = f(resumen anterior, turnos que salen). Se actualiza de a bloques de cuatro turnos, no de a uno, para re-resumir menos veces. Lo viejo queda como resumen y lo reciente, textual: es el **summary buffer**. El resumen va arriba, en el mensaje de sistema, y tiene su propio presupuesto de palabras.

In [47]:
PROMPT_RESUMEN = """Resumen de la conversación hasta ahora:
{resumen}

Turnos nuevos:
{turnos}

Escribí el resumen actualizado de toda la conversación, en no más de {palabras} palabras."""

def actualizar_resumen(resumen, turnos, palabras=PALABRAS_RESUMEN):
    return ask(PROMPT_RESUMEN.format(resumen=resumen or "(todavía no hay resumen)",
                                     turnos=texto_de(turnos), palabras=palabras),
               num_predict=400)

RESUMENES = []                                 # cada version, para ver como cambia
resumen = ""
for j in range(0, N_AFUERA, 4):
    bloque = HIST[2 * j: 2 * min(j + 4, N_AFUERA)]
    resumen = actualizar_resumen(resumen, bloque)
    RESUMENES.append((j, min(j + 4, N_AFUERA) - 1, resumen))
    print("resumen después de los turnos %d–%d (%d tokens)" % (j, min(j + 4, N_AFUERA) - 1, contar(resumen)))
RESUMEN_CORRIDO = resumen
print()
mostrar("EL RESUMEN DE LOS TURNOS 0–%d" % (N_AFUERA - 1), RESUMEN_CORRIDO)
print("¿conserva «Vigo»?          ", "sí" if "vigo" in RESUMEN_CORRIDO.lower() else "NO")
print("¿conserva «1256/2011»?     ", "sí" if "1256/2011" in RESUMEN_CORRIDO else "NO")
print("¿conserva «artículo 9»?    ", "sí" if re.search(r"art[ií]culo\s*9\b", RESUMEN_CORRIDO, re.I) else "NO")

resumen después de los turnos 0–3 (131 tokens)
resumen después de los turnos 4–7 (176 tokens)
resumen después de los turnos 8–11 (91 tokens)
resumen después de los turnos 12–12 (123 tokens)

──────────────────────────────────────────────────────────────────────────
EL RESUMEN DE LOS TURNOS 0–12
──────────────────────────────────────────────────────────────────────────
La conversación aborda normativas de pesca en el Mar Báltico y el Mar del
Norte, reglas para vuelos de aeronaves matriculadas en la UE, efecto
unitario de patentes europeas, condiciones para afirmaciones sobre el
colesterol en alimentos, ventajas aduaneras de las islas atlánticas
portuguesas, y el Reglamento (UE) No 528/2012, que amplió sustancias
autorizadas en productos biocidas desde el 1 de julio de 2012.

¿conserva «Vigo»?           NO
¿conserva «1256/2011»?      NO
¿conserva «artículo 9»?     NO


### 6.2 · Descargar, no borrar

Lo que sale de la ventana se escribe **entero** en un archivo, y en la ventana queda un puntero. Cuando una pregunta lo necesita, se busca en lo descargado —con el mismo embedder de RAG— y se trae de vuelta: es buscar en la propia conversación, la primera memoria fuera de la ventana.

In [48]:
from pathlib import Path

ARCHIVO = Path("historial_01.md")
PARES_AFUERA = [HIST[2 * j: 2 * j + 2] for j in range(N_AFUERA)]
ARCHIVO.write_text("\n\n".join("## turno %d\n%s" % (j, texto_de(p)) for j, p in enumerate(PARES_AFUERA)),
                   encoding="utf-8")
PUNTERO = "[turnos 0–%d → %s: se pueden traer de vuelta]" % (N_AFUERA - 1, ARCHIVO.name)
V_AFUERA = embeber([texto_de(p) for p in PARES_AFUERA])

def traer_de_vuelta(pregunta, k=2):
    """Los k turnos descargados mas parecidos a la pregunta, con su numero."""
    q = embeber([pregunta])[0]
    orden = np.argsort(V_AFUERA @ q)[::-1][:k]
    return [(int(j), PARES_AFUERA[int(j)]) for j in sorted(orden)]

print("descargado:", ARCHIVO, "·", ARCHIVO.stat().st_size, "bytes ·", N_AFUERA, "turnos")
print("puntero   :", PUNTERO)
for j, par in traer_de_vuelta(PRUEBA_CIUDAD):
    print("  para «%s» vuelve el turno %d: %s" % (PRUEBA_CIUDAD, j, recortar(par[0]["content"], 70)))

descargado: historial_01.md · 6790 bytes · 13 turnos
puntero   : [turnos 0–12 → historial_01.md: se pueden traer de vuelta]
  para «¿En qué ciudad trabajo?» vuelve el turno 0: Antes de empezar, dos cosas: trabajo en una pesquera de Vigo, y necesi…
  para «¿En qué ciudad trabajo?» vuelve el turno 4: ¿Y para 2013?


### 6.3 · Las cuatro apuestas, sobre el mismo hecho

Cada política arma la ventana a su manera, adentro del mismo presupuesto. La columna que importa es la del medio: **¿el dato llega al modelo?** Si no llega, la respuesta no puede tenerlo, por bueno que sea el modelo.

In [49]:
def ventana(politica, pregunta):
    """(mensajes, num_ctx) de cada politica. Todas adentro del presupuesto, salvo el control."""
    if politica == "pegar todo (control)":
        return [{"role": "system", "content": SISTEMA}] + HIST + \
               [{"role": "user", "content": pregunta}], NUM_CTX
    if politica == "recortar":
        m, _, _ = ensamblar(pregunta, HIST, [], tope_historial=TOPE_HISTORIAL)
    elif politica == "anclar el turno 0":
        m, _, _ = ensamblar(pregunta, HIST, [], tope_historial=TOPE_HISTORIAL, anclas=1)
    elif politica == "resumir":
        sis = SISTEMA + "\n\n<resumen de la conversación anterior>\n%s\n</resumen de la conversación anterior>" % RESUMEN_CORRIDO
        m, _, _ = ensamblar(pregunta, RECIENTES, [], sistema=sis, tope_historial=TOPE_HISTORIAL)
    elif politica == "descargar":
        vuelta = traer_de_vuelta(pregunta)
        memoria = PUNTERO + "\n" + "\n".join("turno %d · %s" % (j, texto_de(p)) for j, p in vuelta)
        m, _, _ = ensamblar(pregunta, RECIENTES, [], memoria=memoria, tope_historial=TOPE_HISTORIAL)
    return m, PRESUPUESTO

POLITICAS = ["pegar todo (control)", "recortar", "anclar el turno 0", "resumir", "descargar"]

def probar(pregunta, claves):
    """Corre la pregunta con cada politica. claves: {nombre: regex} de los datos que tienen
    que llegar al modelo. Devuelve (filas, columnas) para tabla()."""
    filas = []
    for pol in POLITICAS:
        msgs, ctx = ventana(pol, pregunta)
        adentro = "\n".join(m["content"] for m in msgs[:-1])
        obj, r = estructurado(msgs, RespuestaChat, num_predict=500, num_ctx=ctx)
        resp = solo_respuesta(obj, r)
        fila = {"tokens en la ventana": contar_prompt(msgs)}
        for nombre, patron in claves.items():
            fila["¿«%s» llega?" % nombre] = "sí" if re.search(patron, adentro, re.I) else "no"
        fila["respuesta"] = recortar(resp, 70)
        filas.append((pol, fila))
    columnas = ["tokens en la ventana"] + ["¿«%s» llega?" % n for n in claves] + ["respuesta"]
    return filas, columnas

filas_ciudad, cols = probar(PRUEBA_CIUDAD, {"Vigo": r"vigo"})
tabla("«%s» · el dato se dijo en el turno 0" % PRUEBA_CIUDAD, cols, filas_ciudad)
RESULTADOS["6 · olvidar a propósito"] = dict(filas_ciudad)


«¿En qué ciudad trabajo?» · el dato se dijo en el turno 0
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                         tokens en la ventana   ¿«Vigo» llega?                                                                 respuesta
pegar todo (control)                     4207               sí                                                         Trabajas en Vigo.
recortar                                 1762               no         No se ha proporcionado información sobre la ciudad donde trabajo.
anclar el turno 0                        1599               sí                                                         Trabajas en Vigo.
resumir                                  1903               no         No se ha proporcionado información sobre la ciudad donde trabajo.
descargar                                2118               sí   Acuso recibo. Estoy aquí para ayudarte con la normativ

**Cómo leerlo.** Recortar es gratis y predecible, y olvida justo lo que se dijo primero. Anclar el turno 0 lo emparcha para este dato, no para el que vino después. El resumen conserva lo que el resumidor creyó importante: mejor apuesta que recortar, y sigue siendo una apuesta. Descargar no apostó: el turno entero estaba en el archivo, y si la búsqueda lo trae de vuelta, llega textual —su único riesgo es el de cualquier RAG, que la búsqueda no lo encuentre—. Que llegue tampoco quiere decir que se use bien: la respuesta arranca copiando la que el modelo dio en el turno 0 («Acuso recibo…»). Las políticas también cuestan distinto: resumir pagó una llamada por bloque, y cada vez que el resumen cambia, cambia el principio del prompt y se rompe el cache de prefijos.

Dos láminas de esta sección son mecanismos que no corren en este stack —el umbral de MemGPT para compactar y la compresión de pasajes— y una ya está hecha por diseño: los pasajes nunca entraron al historial, así que no hay nada que **limpiar** antes de resumir.

▶ **Láminas 35–41** (olvidar es una decisión; recortar; resumir; descargar). Sin celda: 38, 39 y 40.

## 7 · Resumir por partes

*Momento 3 · Cuando ya no entra (sigue)*

Hasta acá resumimos una conversación. El mismo problema aparece con un documento, o un corpus entero, que no entra en la ventana. Y ahí aparece una clase de pregunta que RAG no sabe contestar: la **pregunta global**, cuya respuesta está repartida en todo el corpus.

In [50]:
RE_PESCA = re.compile(r"\bpesca|pesquer|\bcapturas?\b", re.IGNORECASE)
ACTOS_PESCA = [c for c in TEXTO_DE
               if RE_PESCA.search(TITULO_DE[c]) and "carbono" not in TITULO_DE[c].lower()]
PREGUNTA_GLOBAL = "¿Cuáles son los temas principales de las normas de pesca de este corpus?"

s_global = responder(PREGUNTA_GLOBAL)
print("actos de pesca en el corpus:", len(ACTOS_PESCA))
print("actos que leyó RAG          :", len(a_documentos(s_global["pasajes"])),
      "→", " ".join(a_documentos(s_global["pasajes"])))
mostrar("CON RAG · top-%d" % K, s_global["respuesta"])

actos de pesca en el corpus: 32
actos que leyó RAG          : 5 → 32010R0237 32011R1343 32014D0555 32011D0207 32010R1089
──────────────────────────────────────────────────────────────────────────
CON RAG · top-5
──────────────────────────────────────────────────────────────────────────
Los temas principales de las normas de pesca en este corpus son la gestión
del esfuerzo pesquero, los permisos de pesca especiales, el control y la
inspección de la pesca, y las medidas técnicas específicas para especies
como el atún rojo. [32010R0237 · encabezado], [32011R1343 · Artículo 1],
[32011D0207 · Artículo 4]



**Top-k trae k fragmentos de una esquina, y el generador resume esa esquina como si fuera el todo.** Lo que se recupera era el techo de la respuesta, y acá el techo es un rincón.

### 7.1 · Map-reduce sobre el corpus

Si el texto no entra, se resume por partes. **Map-reduce**: se parte el material, se resume cada parte por separado (el *map*) y después se resumen los resúmenes (el *reduce*). Acá las partes son fichas de los actos de pesca —título y primer artículo—, de a ocho por llamada.

In [51]:
def ficha(c, tope=220):
    """Titulo y primer articulo de un acto, cortados a 'tope' tokens del generador."""
    primero = next((cuerpo for et, cuerpo in partir_en_articulos(TEXTO_DE[c]) if et != "encabezado"), "")
    texto = "%s\n%s" % (TITULO_DE[c], re.sub(r"\s+", " ", primero))
    return "[%s] %s" % (c, TOK_GEN.decode(TOK_GEN.encode(texto, add_special_tokens=False)[:tope]))

MAP_GLOBAL = """Estos son actos de la normativa pesquera europea:

{bloque}

Listá los temas que tratan, en viñetas cortas, con los identificadores entre corchetes de los
actos de cada tema."""

REDUCE_GLOBAL = """Estas son listas parciales de temas de las normas de pesca de un corpus:

{parciales}

{pregunta} Juntá los temas repetidos, decí cuántos actos tiene cada uno y respondé en viñetas."""

t0 = time.time()
bloques = [ACTOS_PESCA[j:j + 8] for j in range(0, len(ACTOS_PESCA), 8)]
parciales = [ask(MAP_GLOBAL.format(bloque="\n\n".join(ficha(c) for c in b)), num_predict=500)
             for b in bloques]
final_global = ask(REDUCE_GLOBAL.format(parciales="\n\n".join(parciales), pregunta=PREGUNTA_GLOBAL),
                   num_predict=700)
print("map-reduce: %d actos · %d llamadas de map + 1 de reduce · %.0f s"
      % (len(ACTOS_PESCA), len(bloques), time.time() - t0))
mostrar("CON MAP-REDUCE SOBRE LOS %d ACTOS DE PESCA" % len(ACTOS_PESCA), final_global)
RESULTADOS["7 · pregunta global"] = {"actos RAG": len(a_documentos(s_global["pasajes"])),
                                     "actos map-reduce": len(ACTOS_PESCA),
                                     "llamadas map-reduce": len(bloques) + 1}

map-reduce: 32 actos · 4 llamadas de map + 1 de reduce · 69 s
──────────────────────────────────────────────────────────────────────────
CON MAP-REDUCE SOBRE LOS 32 ACTOS DE PESCA
──────────────────────────────────────────────────────────────────────────
- **Posibilidades de pesca en el Mar Báltico**: 5 actos
- **Posibilidades de pesca para peces de aguas profundas**: 3 actos
- **Participación financiera de la UE en programas de control pesquero**:
4 actos
- **Plan de descartes para pesquerías pelágicas**: 5 actos
- **Posibilidades de pesca para peces sujetos a acuerdos
internacionales**: 1 acto
- **Posibilidades de pesca para peces no sujetos a acuerdos
internacionales**: 1 acto
- **Normas de control e inspección para pesquerías pelágicas**: 3 actos
- **Normas de aplicación de medidas de conservación y de la CGPM**: 2
actos
- **Posibilidades de pesca en 2013 y 2014**: 2 actos
- **Posibilidades de pesca para 2012 y 2013**: 2 actos
- **Normas sanitarias para productos de la pesca de Gro

### 7.2 · Tres cadenas sobre un acto largo

El acto con el que arrancaba el corpus de la clase de RAG, `32015R0009`, no entraba en el embedder. Lo resumimos de tres formas canónicas: **stuff** (todo en una llamada, si entra: el «si entra, se pega» de siempre), **map-reduce** (cada pedazo por separado, y después juntar) y **refine** (en orden, arrastrando un resumen que cada pedazo actualiza: la misma operación que el resumen corrido de la sección 6).

Lo que se compara es **qué sobrevive**: un puñado de datos del acto, unos en los considerandos y otros en los artículos. Uno de ellos es una relación entre partes: el punto 6 del artículo 1 suprime el apartado 3 del artículo 36 del reglamento que modifica, y el porqué —una excepción transitoria que pasa a ser permanente— está en los considerandos, muchas páginas antes.

In [52]:
ACTO_LARGO = "32015R0009"
texto_largo = TEXTO_DE.get(ACTO_LARGO, "")
print("%s · %d tokens del embedder (el pipeline corta en %d) · %d del generador"
      % (ACTO_LARGO, contar_emb(texto_largo), embedder.max_seq_length, contar(texto_largo)))

def trozar(texto, tam=1100):
    ids = TOK_GEN.encode(texto, add_special_tokens=False)
    return [TOK_GEN.decode(ids[j:j + tam]) for j in range(0, len(ids), tam)]

PIEZAS = trozar(texto_largo)
PIDE = "en un párrafo de no más de 120 palabras: qué modifica, por qué, y desde cuándo"

CADENAS = {}
t0 = time.time()
if contar(texto_largo) + 700 < NUM_CTX:
    CADENAS["stuff"] = (ask("Resumí este acto %s.\n\n%s" % (PIDE, texto_largo), num_predict=400), 1)
else:
    CADENAS["stuff"] = ("(no entra en la ventana)", 0)

parciales = [ask("Resumí este fragmento de un acto en dos o tres oraciones.\n\n%s" % p, num_predict=200)
             for p in PIEZAS]
CADENAS["map-reduce"] = (ask("Estos son resúmenes de fragmentos consecutivos de un mismo acto:\n\n%s\n\n"
                             "Escribí un resumen del acto entero %s." % ("\n\n".join(parciales), PIDE),
                             num_predict=400), len(PIEZAS) + 1)

res = ""
for p in PIEZAS:
    res = ask("Resumen del acto hasta acá:\n%s\n\nFragmento siguiente:\n%s\n\nActualizá el resumen con "
              "el fragmento nuevo, %s." % (res or "(todavía no hay resumen)", p, PIDE), num_predict=400)
CADENAS["refine"] = (res, len(PIEZAS))
print("%d pedazos · tres cadenas en %.0f s" % (len(PIEZAS), time.time() - t0))

DATOS = {"modifica el 142/2011": r"142/2011",
         "la excepción pasa a ser permanente": r"permanente",
         "suprime el art. 36.3": r"\b36\b",
         "hasta 20 kg por semana": r"20\s*kg",
         "sustratos: < 5 % y < 50 %": r"5\s*%.*50\s*%|50\s*%.*5\s*%",
         "transición hasta el 27/9/2015": r"27 de septiembre",
         "aplicable desde el 23/2/2015": r"23 de febrero"}
filas = []
for nombre, (resumen_c, llamadas) in CADENAS.items():
    fila = {"llamadas": llamadas}
    fila.update({d: "✓" if re.search(p, resumen_c, re.I | re.S) else "·" for d, p in DATOS.items()})
    filas.append((nombre, fila))
tabla("Qué sobrevivió en cada resumen", ["llamadas"] + list(DATOS), filas)
for nombre, (resumen_c, _) in CADENAS.items():
    mostrar(nombre.upper(), resumen_c)
RESULTADOS["7 · cadenas"] = dict(filas)

32015R0009 · 5357 tokens del embedder (el pipeline corta en 1024) · 6752 del generador
7 pedazos · tres cadenas en 102 s

Qué sobrevivió en cada resumen
──────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
               llamadas   modifica el 142/2011   la excepción pasa a ser permanente   suprime el art. 36.3   hasta 20 kg por semana   sustratos: < 5 % y < 50 %   transición hasta el 27/9/2015   aplicable desde el 23/2/2015
stuff                 1                      ✓                                    ·                      ·                        ·                           ·                               ·                              ✓
map-reduce            8                      ✓                                    ·                      ·                        ·                           ·                   

**Cómo leerlo.** Map-reduce escala porque cada pedazo se resume solo, y por eso mismo pierde lo que vivía **entre** pedazos y pierde la proporción: cada pedazo pesa lo mismo en el reduce. Refine sabe lo que vino antes, así que una relación puede sobrevivir; a cambio, lo del principio se recomprime en cada paso. Un árbol de resúmenes —resúmenes de resúmenes, como RAPTOR— y los resúmenes por comunidades de GraphRAG llevan esta idea un paso más allá; ordenarlo por entidades y relaciones es el momento 5, datos conectados, en el notebook B.

*Nota de laboratorio:* en LlamaIndex estas cadenas se llaman *compact*, *tree_summarize* y *refine*; su modo *simple_summarize* recorta en silencio lo que no entra.

▶ **Láminas 42–46** (la pregunta que ningún pasaje responde; las tres cadenas; map-reduce; refine; resúmenes de resúmenes).

## 8 · Lo que el resumen pierde

*Momento 3 · Cuando ya no entra (sigue)*

Todas las políticas de olvido deciden qué importa **en el momento de sacar algo de la ventana**, y en ese momento la pregunta que lo va a necesitar todavía no existe. Es la falla del vector precalculado de la clase de RAG, sin el texto original para volver a leer.

En el turno 7 el usuario dejó un dato preciso: el reglamento y el artículo que va a citar. El resumen corrido de la sección 6 lo cubrió. Ahora la pregunta llega.

In [53]:
print("el turno 7, textual:", GUION[7]["turno"])
print()
for j_ini, j_fin, version in RESUMENES:
    tiene = "1256/2011" in version and re.search(r"art[ií]culo\s*9\b", version, re.I)
    print("resumen tras los turnos %2d–%2d · %3d tokens · ¿«1256/2011, artículo 9»? %s"
          % (j_ini, j_fin, contar(version), "sí" if tiene else "NO"))
print()
filas_informe, cols = probar(PRUEBA_INFORME, {"1256/2011": r"1256/2011",
                                              "artículo 9": r"art[ií]culo\s*9\b"})
tabla("«%s» · el dato se dijo en el turno 7" % PRUEBA_INFORME, cols, filas_informe)
RESULTADOS["8 · el identificador"] = dict(filas_informe)

el turno 7, textual: Anotá esto para el informe: el que voy a citar es el Reglamento (UE) nº 1256/2011, artículo 9.

resumen tras los turnos  0– 3 · 131 tokens · ¿«1256/2011, artículo 9»? NO
resumen tras los turnos  4– 7 · 176 tokens · ¿«1256/2011, artículo 9»? NO
resumen tras los turnos  8–11 ·  91 tokens · ¿«1256/2011, artículo 9»? NO
resumen tras los turnos 12–12 · 123 tokens · ¿«1256/2011, artículo 9»? NO


«¿Qué reglamento y qué artículo te dije que iba a citar en el informe?» · el dato se dijo en el turno 7
────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────────
                         tokens en la ventana   ¿«1256/2011» llega?   ¿«artículo 9» llega?                                                                 respuesta
pegar todo (control)                     4223                    sí                     sí   El Reglamento (UE) nº 1256/2011, artículo 9, es el que vas

**Lo que hay que ver.** Si el resumen perdió el número —un identificador, un artículo: lo primero que se va—, ya no está, y no hay nada que releer. Descargar con un puntero es la única política que no apostó, y por eso la única que puede devolverlo. Cuando el resumen se reescribe una y otra vez, además, se gasta: cada vuelta es más corta y más genérica (*context collapse*). La columna de tokens de las versiones del resumen, arriba, es una vista chica de eso.

> **Estamos acá · fin del momento 3 y de la semana 9.** Una conversación que funciona: en chico, lo que el harness de Claude hace en cada mensaje. Y **tres deudas con nombre**:
>
> - Los resúmenes pierden e inventan, y deciden antes de la pregunta.
> - Cada técnica cuesta llamadas y rompe caches.
> - La que queda abierta: **todo vivió adentro de la sesión**. El historial, el resumen y hasta el archivo descargado quedan en esta conversación; la próxima empieza de cero.
>
> Claude, en un chat nuevo, se acuerda de lo que se le contó otro día: cómo lo hace es la semana que viene, desde la sección 9.

▶ **Láminas 47–50** (el resumen se escribió antes de la pregunta; cómo se equivoca un resumen; resumir el resumen; tres deudas con nombre).

## 9 · Al día siguiente

*Momento 4 · Al día siguiente · desde acá, la semana 10*

**Memoria de largo plazo.** Quedó abierta una deuda: la sesión se cerró, y con ella el historial, el resumen y lo descargado. Desde acá, la vida de la conversación sigue en **sesiones**, con fecha. Son tres, cortas, del mismo usuario; las preguntas otra vez escritas a mano y las respuestas, del pipeline. Lo que importa son los datos que el usuario deja en el camino:

- **Sesión 1 (marzo):** trabaja en una pesquera de Vigo, sigue el Báltico, quiere el artículo citado, y el arrastrero se llama *Santa Lucía*.
- **Sesión 2 (mayo):** consulta el Mar del Norte.
- **Sesión 3 (agosto):** se pasó a una pesquera de Gdansk; sigue con el Báltico.

La sesión 4 es «hoy», y es la de las preguntas.

In [54]:
HOY = datetime.date(2026, 9, 29)       # la fecha de la sesion 4

SESIONES = [
    {"id": "S1", "fecha": datetime.date(2026, 3, 3), "turnos": [
        "Hola. Trabajo en una pesquera de Vigo y sigo las cuotas del Mar Báltico. Las respuestas, "
        "siempre con el artículo citado.",
        "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2012?",
        "¿Y qué dice sobre el bacalao?",
        "Nuestro arrastrero se llama Santa Lucía; lo vamos a matricular este año."]},
    {"id": "S2", "fecha": datetime.date(2026, 5, 12), "turnos": [
        "Hoy me interesa el Mar del Norte: ¿cuándo se puede cerrar una pesquería de un día para otro?",
        "¿A qué especies alcanza esa medida?"]},
    {"id": "S3", "fecha": datetime.date(2026, 8, 25), "turnos": [
        "Te actualizo: me pasé a una pesquera de Gdansk. Sigo con las cuotas del Báltico, eso no cambió.",
        "¿Qué reglamento fijó las posibilidades de pesca del Mar Báltico para 2013?"]},
]

def correr_sesion(s):
    """Una sesion entera, con historial propio que empieza vacio. Lo que no pregunta, no busca."""
    hist = []
    for turno in s["turnos"]:
        es_pregunta = "?" in turno
        consulta = reescribir(hist, turno) if (es_pregunta and hist) else None
        salida = responder(turno, hist, consulta=consulta, pasajes=None if es_pregunta else [])
        al_historial(hist, turno, salida)
    return hist

for s in SESIONES:
    s["historial"] = correr_sesion(s)
    print("sesión %s · %s · %d turnos · %d tokens"
          % (s["id"], s["fecha"], len(s["turnos"]), contar(texto_de(s["historial"]))))

sesión S1 · 2026-03-03 · 4 turnos · 425 tokens
sesión S2 · 2026-05-12 · 2 turnos · 379 tokens
sesión S3 · 2026-08-25 · 2 turnos · 189 tokens


**La sesión 4 arranca.** El usuario retoma donde dejó: «¿Y al final, qué cambió para el bacalao?». Sin nada escrito afuera, el sistema le responde como a un desconocido.

In [55]:
PREGUNTA_S4 = "¿Y al final, qué cambió para el bacalao?"
s4 = responder(PREGUNTA_S4)          # sesion nueva: sin historial y sin memoria
mostrar("SESIÓN 4 · «%s» · sin memoria" % PREGUNTA_S4, s4["respuesta"])
print("actos que trajo el buscador:", " ".join(a_documentos(s4["pasajes"])))

──────────────────────────────────────────────────────────────────────────
SESIÓN 4 · «¿Y al final, qué cambió para el bacalao?» · sin memoria
──────────────────────────────────────────────────────────────────────────
Se adaptó el TAC del bacalao en el Mar Céltico a un nuevo dictamen
científico para 2011 y se introdujo flexibilidad en la gestión del
esfuerzo pesquero para el bacalao del Mar Báltico. Además, se excluyeron a
algunos grupos de buques del régimen de gestión del esfuerzo pesquero si
cumplían ciertos criterios.

actos que trajo el buscador: 32010R0712 32011R1106 32011R1256


### 9.1 · Dos vallas: no recordar y re-leer todo

Antes de construir nada, las dos vallas. El piso es **sin memoria**. La otra es **re-leer todo**: volver a mandar las sesiones pasadas enteras, el «si entra, se pega» entre sesiones. Funciona mientras entran. La tabla cuenta cuánto mide re-leer todo sesión a sesión, contra el presupuesto de la ventana.

In [56]:
def transcripcion(s):
    return "Sesión %s (%s):\n%s" % (s["id"], s["fecha"].isoformat(), texto_de(s["historial"]))

TODO = "\n\n".join(transcripcion(s) for s in SESIONES)
acum, filas = 0, []
for s in SESIONES:
    n = contar(transcripcion(s))
    acum += n
    filas.append((s["id"], {"tokens de la sesión": n, "re-leer todo": acum,
                            "¿entra en %d?" % PRESUPUESTO: "sí" if acum + RESERVA < PRESUPUESTO else "no"}))
tabla("Re-leer todas las sesiones pasadas", ["tokens de la sesión", "re-leer todo",
                                             "¿entra en %d?" % PRESUPUESTO], filas)

por_sesion = COSTO[-1]["historial"]     # una sesion como la conversacion de las secciones 1 a 8
print("Estas sesiones son cortas. Si cada una midiera como la conversación de las secciones 1 a 8")
print("(%d tokens de historial), re-leer todo dejaría de entrar en la ventana de %d en la sesión %d."
      % (por_sesion, PRESUPUESTO, int((PRESUPUESTO - RESERVA) // max(1, por_sesion)) + 1))
RESULTADOS["9 · re-leer todo"] = {"tokens de las tres sesiones": acum}


Re-leer todas las sesiones pasadas
───────────────────────────────────────────────────────────────────
               tokens de la sesión   re-leer todo   ¿entra en 4096?
S1                             442            442                sí
S2                             396            838                sí
S3                             206           1044                sí

Estas sesiones son cortas. Si cada una midiera como la conversación de las secciones 1 a 8
(3754 tokens de historial), re-leer todo dejaría de entrar en la ventana de 4096 en la sesión 1.


**La regla que se sigue: declarar el régimen.** Acá las tres sesiones entran enteras en la ventana, así que re-leer todo es una opción, y la vara contra la que se mide la memoria. Cuando dejan de entrar, re-leer todo ni siquiera es una opción.

**Leer la memoria es RAG; lo nuevo es escribirla.** La memoria de largo plazo es un corpus que escribe el sistema, mientras conversa, con el mismo modelo que habla. Puede estar mal, duplicado o viejo. Las secciones 10 y 11 escriben; la 12 lee; la 13 mide.

▶ **Láminas 51–55** (al día siguiente; las dos vallas; escribir y leer; dónde vive la memoria; traerla justo a tiempo).

## 10 · Qué se escribe

*Momento 4 · Al día siguiente (sigue)*

Una conversación mezcla tres cosas: **lo que pasó** (memoria episódica), **lo que es cierto** (semántica) y **cómo hay que actuar** (procedural). Escribimos las tres, y la extracción es una llamada con un schema: el modelo lee la sesión y devuelve hechos candidatos con campos fijos. Los campos son decisiones de diseño: de qué trata cada hecho, cuánto importa, y —el que más importa— **si lo dijo el usuario o lo infirió el modelo**.

La regla de escritura: **se guarda sólo lo que se quiere conservar**. Ni el campo de análisis, ni lo inferido, ni lo que venga con un «probablemente».

In [57]:
import unicodedata

class Hecho(BaseModel):
    texto: str = Field(description="El hecho, en una oración, en tercera persona: 'El usuario ...'.")
    sujeto: str = Field(description="De qué trata, en una o dos palabras: empleador, región, embarcación, formato...")
    tipo: Literal["semántica", "procedural"] = Field(
        description="semántica: algo cierto sobre el usuario; procedural: cómo quiere que se le responda.")
    importancia: int = Field(description="De 1 (trivial) a 10 (central para atenderlo bien).")
    origen: Literal["lo dijo el usuario", "lo infiero yo"]

class Extraccion(BaseModel):
    analisis: str = Field(description="Qué se dijo en la sesión que valga la pena recordar, en dos oraciones.")
    hechos: list[Hecho]

class Episodio(BaseModel):
    resumen: str = Field(description="Qué consultó el usuario en la sesión, en una oración.")

PROMPT_EXTRAER = """Sesión del {fecha}:
{transcripcion}

Extraé los hechos sobre el USUARIO que valga la pena recordar en sesiones futuras: quién es,
dónde trabaja, qué sigue, cómo quiere las respuestas, nombres propios que mencionó. Sólo lo que
el usuario contó de sí mismo: no lo que dicen los reglamentos ni lo que respondió el asistente.
Un hecho por registro, con su importancia de 1 a 10. No guardes como hechos las preguntas que
hizo el usuario ni lo que consultó: eso es un episodio, y se guarda aparte. Procedural es sólo
cómo quiere que se le responda. Si algo no lo dijo el usuario y lo deducís vos, marcalo como
"lo infiero yo"."""

DUDA = re.compile(r"probablemente|quizás|quizá|posiblemente|parece|podría|supongo", re.IGNORECASE)

def extraer(s):
    """(guardados, descartados con su motivo, analisis) de una sesion."""
    obj, r = estructurado([{"role": "user", "content": PROMPT_EXTRAER.format(
        fecha=s["fecha"].isoformat(), transcripcion=texto_de(s["historial"]))}],
        Extraccion, num_predict=900)
    if obj is None:
        return [], [("(la extracción no parseó)", recortar(r["texto"], 80))], ""
    guardar, descartar = [], []
    for h in obj.hechos:
        if h.origen != "lo dijo el usuario":
            descartar.append((h.texto, "inferido"))
        elif DUDA.search(h.texto):
            descartar.append((h.texto, "con duda"))
        else:
            guardar.append(h)
    return guardar, descartar, obj.analisis

def episodio(s):
    obj, _ = estructurado([{"role": "user", "content": "Sesión del %s:\n%s\n\nResumí qué consultó el usuario."
                            % (s["fecha"].isoformat(), texto_de(s["historial"]))}], Episodio, num_predict=150)
    return obj.resumen if obj else "(sin resumen)"

LLAMADAS_ESCRITURA = 0
for s in SESIONES:
    s["hechos"], s["descartados"], analisis = extraer(s)
    s["episodio"] = episodio(s)
    LLAMADAS_ESCRITURA += 2
    print("SESIÓN %s · %s" % (s["id"], s["fecha"]))
    print("  análisis (no se guarda): %s" % recortar(analisis, 90))
    for h in s["hechos"]:
        print("  + [%-10s · %-10s · %2d] %s" % (h.tipo, recortar(h.sujeto, 10), h.importancia, h.texto))
    for texto, motivo in s["descartados"]:
        print("  − descartado (%s): %s" % (motivo, texto))
    print("  episodio: %s" % s["episodio"])
    print()

SESIÓN S1 · 2026-03-03
  análisis (no se guarda): Los hechos sobre el usuario que valen la pena recordar para futuras sesiones se resumen a …
  + [semántica  · Usuario    ·  9] El usuario trabaja en una pesquera de Vigo.
  + [semántica  · Usuario    ·  8] El usuario sigue las cuotas del Mar Báltico.
  + [procedural · Usuario    · 10] El usuario quiere que las respuestas siempre incluyan el artículo citado.
  + [semántica  · Usuario    ·  7] El usuario mencionó el nombre del arrastrero: Santa Lucía.
  + [semántica  · Usuario    ·  7] El usuario mencionó que va a matricular el arrastrero este año.
  episodio: El usuario consultó sobre el reglamento que fijó las posibilidades de pesca del Mar Báltico para 2012, específicamente sobre el bacalao, y mencionó que su arrastrero, llamado Santa Lucía, será matriculado este año, preguntando sobre los requisitos de matrícula en el contexto de la normativa europea aplicable a su actividad en Vigo.

SESIÓN S2 · 2026-05-12
  análisis (no se guarda): 

**Lo que hay que ver.** El extractor es un modelo y escribe como un modelo: puede guardar una conjetura como si fuera un hecho. Acá lo filtró un campo del schema, y sólo porque el modelo fue honesto al llenarlo; si no lo hubiera sido, la conjetura estaría guardada y se leería en cada sesión siguiente como algo que dijo el usuario. Por eso una memoria se audita: contar, sobre una muestra, cuántos registros están mal o nunca se dijeron. Ese número es parte de evaluarla.

**Cuándo se escribe.** Acá extrajimos después de cada sesión: escritura **en background**, cuidadosa y tardía. La alternativa es escribir **en el momento** (*hot path*), turno a turno: la memoria queda disponible enseguida y el turno se hace más lento.

▶ **Láminas 56–62** (qué vale la pena guardar; los cuatro tipos de memoria; episódica; semántica; procedural; guardar sólo lo que se quiere conservar; cuándo se escribe).

## 11 · Mantener la memoria al día

*Momento 4 · Al día siguiente (sigue)*

Escribir una vez es fácil; lo difícil es que la memoria **siga siendo cierta**. Primero, la falla: una memoria que sólo agrega. Guardamos los hechos de las tres sesiones uno detrás del otro, y preguntamos dónde trabaja el usuario.

In [58]:
def nuevo_registro(texto, sujeto, tipo, importancia, sesion, fecha, origen="lo dijo el usuario"):
    """Una fila de la memoria. Dos relojes: cuando se REGISTRO, y desde/hasta cuando fue CIERTO."""
    return {"id": None, "texto": texto, "sujeto": sujeto, "tipo": tipo,
            "importancia": int(min(10, max(1, importancia))), "sesion": sesion,
            "registrado": fecha, "valido_desde": fecha, "valido_hasta": None, "origen": origen,
            "escrito": time.time(), "vector": embeber([texto])[0]}

def cercanos(pregunta, registros, k=3):
    q = embeber([pregunta])[0]
    return sorted(registros, key=lambda r: -float(r["vector"] @ q))[:k]

SOLO_AGREGAR = []
for s in SESIONES:
    for h in s["hechos"]:
        SOLO_AGREGAR.append(nuevo_registro(h.texto, h.sujeto, h.tipo, h.importancia, s["id"], s["fecha"]))

print("memoria que sólo agrega: %d registros" % len(SOLO_AGREGAR))
print("«¿Dónde trabajo?» trae:")
for r in cercanos("¿Dónde trabajo?", SOLO_AGREGAR):
    print("   [%s] %s" % (r["sesion"], r["texto"]))

memoria que sólo agrega: 10 registros
«¿Dónde trabajo?» trae:
   [S1] El usuario trabaja en una pesquera de Vigo.
   [S3] El usuario se mudó a una pesquera de Gdansk.
   [S2] El usuario se interesa por el Mar del Norte.


**Dos hechos que chocan en la misma ventana** (*context clash*), y nada que diga cuál vale. Con más sesiones, además, el mismo hecho dicho de cinco formas son cinco registros que se reparten el top-k.

### 11.1 · Agregar, actualizar, borrar o nada

Antes de guardar, comparar con lo que ya hay. Por cada hecho candidato se recuperan las memorias vigentes más parecidas, y un modelo elige una de cuatro operaciones: **ADD**, **UPDATE**, **DELETE** o **NOOP**. Es la **consolidación** (*consolidation*), y es un structured output con su orden: primero el análisis, después la operación, después a qué registro afecta.

Y una decisión de diseño más: **invalidar, no borrar**. Cada registro lleva dos relojes —cuándo se registró, y desde y hasta cuándo fue cierto—. Un UPDATE o un DELETE no borran el registro viejo: le ponen fecha de fin. Es un registro **bitemporal**, en dos columnas de fecha.

In [59]:
class Decision(BaseModel):
    analisis: str = Field(description="Cómo se relaciona el hecho nuevo con los registros parecidos, en una oración.")
    operacion: Literal["ADD", "UPDATE", "DELETE", "NOOP"] = Field(
        description="ADD: es nuevo. UPDATE: reemplaza o corrige un registro. DELETE: un registro dejó de "
                    "ser cierto y el hecho nuevo no lo reemplaza. NOOP: ya estaba.")
    registro: str = Field(description="El id del registro afectado para UPDATE o DELETE; vacío para ADD y NOOP.")

PROMPT_CONSOLIDAR = """Registros de memoria vigentes, parecidos al hecho nuevo:
{similares}

Hecho nuevo (sesión del {fecha}): {hecho}

Decidí qué hacer con la memoria."""

MEMORIA, BITACORA = [], []

def vigentes(fecha=None):
    """Los registros ciertos en 'fecha' (por default, los que no tienen fecha de fin)."""
    if fecha is None:
        return [r for r in MEMORIA if r["valido_hasta"] is None]
    return [r for r in MEMORIA if r["valido_desde"] <= fecha and
            (r["valido_hasta"] is None or fecha < r["valido_hasta"])]

def agregar(reg):
    reg["id"] = "m%d" % (len(MEMORIA) + 1)
    MEMORIA.append(reg)
    return reg["id"]

def consolidar(h, s):
    global LLAMADAS_ESCRITURA
    nuevo = nuevo_registro(h.texto, h.sujeto, h.tipo, h.importancia, s["id"], s["fecha"])
    parecidos = [r for r in cercanos(h.texto, vigentes()) if r["tipo"] != "episódica"]
    if not parecidos:
        BITACORA.append((s["id"], h.texto, "ADD", agregar(nuevo), "(memoria vacía: no hay con qué comparar)"))
        return
    lista = "\n".join("[%s] %s (desde %s)" % (r["id"], r["texto"], r["valido_desde"]) for r in parecidos)
    obj, _ = estructurado([{"role": "user", "content": PROMPT_CONSOLIDAR.format(
        similares=lista, fecha=s["fecha"], hecho=h.texto)}], Decision, num_predict=250)
    LLAMADAS_ESCRITURA += 1
    op = obj.operacion if obj else "ADD"
    viejo = next((r for r in MEMORIA if obj and r["id"] == obj.registro.strip("[] ")), None)
    if op in ("UPDATE", "DELETE") and viejo is not None:
        viejo["valido_hasta"] = s["fecha"]                    # invalidar, no borrar
        rid = agregar(nuevo) if op == "UPDATE" else viejo["id"]
    elif op == "NOOP":
        rid = parecidos[0]["id"]
    else:
        op, rid = "ADD", agregar(nuevo)
    BITACORA.append((s["id"], h.texto, op, rid, obj.analisis if obj else "(no parseó)"))

for s in SESIONES:
    for h in s["hechos"]:
        consolidar(h, s)
    agregar(nuevo_registro("El %s el usuario consultó: %s" % (s["fecha"].isoformat(), s["episodio"]),
                           "consulta", "episódica", 4, s["id"], s["fecha"]))

print("LA BITÁCORA DE ESCRITURA")
for sid, hecho, op, rid, analisis in BITACORA:
    print("  %s · %-6s %-4s · %s" % (sid, op, rid, recortar(hecho, 70)))
    print("                    %s" % recortar(analisis, 90))
print()
print("LA MEMORIA")
for r in MEMORIA:
    print("  %-4s %-10s desde %s hasta %-10s %s" % (r["id"], r["tipo"], r["valido_desde"],
          r["valido_hasta"] or "hoy", recortar(r["texto"], 60)))
RESULTADOS["11 · consolidación"] = {"operaciones": [b[2] for b in BITACORA],
                                    "registros": len(MEMORIA), "vigentes": len(vigentes())}

LA BITÁCORA DE ESCRITURA
  S1 · ADD    m1   · El usuario trabaja en una pesquera de Vigo.
                    (memoria vacía: no hay con qué comparar)
  S1 · ADD    m2   · El usuario sigue las cuotas del Mar Báltico.
                    La memoria [m1] indica que el usuario trabaja en una pesquera de Vigo, lo cual sugiere una…
  S1 · ADD    m3   · El usuario quiere que las respuestas siempre incluyan el artículo cita…
                    El hecho nuevo establece que el usuario quiere que las respuestas siempre incluyan el artí…
  S1 · ADD    m4   · El usuario mencionó el nombre del arrastrero: Santa Lucía.
                    El hecho nuevo menciona que el usuario mencionó el nombre del arrastrero: Santa Lucía. Est…
  S1 · ADD    m5   · El usuario mencionó que va a matricular el arrastrero este año.
                    El hecho nuevo indica que el usuario va a matricular el arrastrero este año. Dado que ya m…
  S2 · ADD    m7   · El usuario se interesa por el Mar del Norte.
           

### 11.2 · Dos relojes, dos preguntas

Con los registros invalidados en vez de borrados, la memoria responde qué es cierto hoy **y** qué era cierto antes. Para que la demo no dependa de lo que eligió el modelo arriba, aplicamos a mano la operación correcta para el cambio de trabajo —un UPDATE: «trabaja en Vigo» se cierra el día de la sesión 3 y «trabaja en Gdansk» arranca ese día— sobre una copia de la memoria, y preguntamos por el empleador hoy y en mayo.

In [60]:
eleccion = [b[2] for b in BITACORA if re.search(r"Gda[nń]sk", b[1])]
print("lo que eligió el modelo para el cambio de trabajo:", ", ".join(eleccion) or "(ningún hecho lo nombra)")

S3 = next(s for s in SESIONES if s["id"] == "S3")
COPIA = [dict(r) for r in MEMORIA if r["tipo"] != "episódica" and not re.search(r"Gda[nń]sk", r["texto"])]
for r in COPIA:
    if "Vigo" in r["texto"]:
        r["valido_hasta"] = S3["fecha"]                      # el UPDATE correcto: cerrar el viejo...
COPIA.append(nuevo_registro("El usuario trabaja en una pesquera de Gdansk.", "Usuario", "semántica", 9,
                            "S3", S3["fecha"]))                 # ...y abrir el nuevo el mismo día

def ciertos(registros, fecha=None):
    if fecha is None:
        return [r for r in registros if r["valido_hasta"] is None]
    return [r for r in registros if r["valido_desde"] <= fecha and
            (r["valido_hasta"] is None or fecha < r["valido_hasta"])]

print()
for etiqueta, fecha in (("hoy", None), ("en mayo", datetime.date(2026, 5, 15))):
    top = cercanos("¿Dónde trabaja el usuario?", ciertos(COPIA, fecha), k=1)
    print("¿dónde trabaja %-8s → %s" % (etiqueta + "?", top[0]["texto"] if top else "(nada)"))
print()
print("Una memoria que borrara el registro viejo respondería bien la primera y no la segunda.")

lo que eligió el modelo para el cambio de trabajo: ADD

¿dónde trabaja hoy?     → El usuario trabaja en una pesquera de Gdansk.
¿dónde trabaja en mayo? → El usuario trabaja en una pesquera de Vigo.

Una memoria que borrara el registro viejo respondería bien la primera y no la segunda.


**Si una de las dos respuestas sale mal, mirá cuál filtro falló.** La fecha decide qué registros eran ciertos ese día —«trabaja en Vigo» lo era en mayo—; entre esos, el más parecido a la pregunta lo elige la similitud, y puede preferir otro registro vigente que habla del usuario pero no de su trabajo. La fecha filtra qué era cierto, no qué es relevante.

**Dos puntos donde fallar.** La consolidación es una recuperación seguida de una decisión: si la búsqueda no trae el registro viejo, se agrega un duplicado; si el modelo decide mal, se pisa lo que no había que pisar. Hay que leer la bitácora, no sólo el resultado. A cambio, es el antídoto del colapso de contexto de la sección 8: se toca un registro, no se reescribe un resumen.

▶ **Láminas 63–65** (el usuario cambió de trabajo; agregar, actualizar, borrar o nada; invalidar, no borrar). La lámina 66, olvidar a largo plazo, no tiene celda.

## 12 · Leer la memoria

*Momento 4 · Al día siguiente (sigue)*

La memoria está escrita y al día. Falta traerla cuando hace falta, y acá la similitud se queda corta: no sabe **cuándo pasó** ni **cuánto importa**. *Generative Agents* lo resolvió con una suma de tres puntajes: **relevancia** (el coseno con la pregunta), **recencia** (un decaimiento de 0,995 por hora) e **importancia** (el número de 1 a 10 que el extractor le puso a cada hecho). Cada término se normaliza a [0, 1] con min-max y se suman.

En el paper, la recencia se mide desde la última vez que la memoria se recordó; acá, sin historia de lecturas, desde que se registró.

In [61]:
def horas(desde, hasta):
    return (datetime.datetime.combine(hasta, datetime.time(12)) -
            datetime.datetime.combine(desde, datetime.time(12))).total_seconds() / 3600

def minmax(xs):
    lo, hi = min(xs), max(xs)
    return [(x - lo) / (hi - lo) if hi > lo else 0.0 for x in xs]

def leer(pregunta, registros, k=5, reloj="escenario"):
    """Los k registros de mayor relevancia + recencia + importancia, con sus terminos crudos.
    reloj='escenario' usa las fechas de las sesiones; reloj='computadora', la hora en que este
    notebook escribio cada registro."""
    q = embeber([pregunta])[0]
    rel = [float(r["vector"] @ q) for r in registros]
    if reloj == "escenario":
        rec = [0.995 ** horas(r["registrado"], HOY) for r in registros]
    else:
        ahora = time.time()
        rec = [0.995 ** ((ahora - r["escrito"]) / 3600) for r in registros]
    imp = [r["importancia"] / 10 for r in registros]
    total = [a + b + c for a, b, c in zip(minmax(rel), minmax(rec), minmax(imp))]
    orden = sorted(range(len(registros)), key=lambda i: -total[i])[:k]
    crudos = {"relevancia": rel, "recencia": rec, "importancia": imp}
    return [dict(registros[i], puntaje=total[i]) for i in orden], crudos

PREGUNTA_LEER = "¿Qué sabés de mi trabajo?"
for reloj in ("escenario", "computadora"):
    elegidos, crudos = leer(PREGUNTA_LEER, MEMORIA, k=3, reloj=reloj)
    tabla("Los tres términos, crudos · reloj: %s" % reloj, ["mín", "máx", "rango"],
          [(t, {"mín": "%.6g" % min(v), "máx": "%.6g" % max(v), "rango": "%.3g" % (max(v) - min(v))})
           for t, v in crudos.items()])
    print("top-3 para «%s» con el reloj de%s:" % (PREGUNTA_LEER, "l escenario" if reloj == "escenario" else " la computadora"))
    for r in elegidos:
        print("   %.2f  [%s] %s" % (r["puntaje"], r["sesion"], recortar(r["texto"], 70)))
RESULTADOS["12 · leer"] = {"rango de recencia (computadora)": max(crudos["recencia"]) - min(crudos["recencia"])}


Los tres términos, crudos · reloj: escenario
───────────────────────────────────────────────
                       mín         máx    rango
relevancia        0.317248    0.456596    0.139
recencia        1.0674e-11   0.0148384   0.0148
importancia            0.4           1      0.6

top-3 para «¿Qué sabés de mi trabajo?» con el reloj del escenario:
   2.69  [S3] El usuario se mudó a una pesquera de Gdansk.
   1.83  [S1] El usuario trabaja en una pesquera de Vigo.
   1.36  [S2] El usuario se interesa por el Mar del Norte.

Los tres términos, crudos · reloj: computadora
──────────────────────────────────────────────
                     mín        máx      rango
relevancia      0.317248   0.456596      0.139
recencia        0.999923   0.999999   7.58e-05
importancia          0.4          1        0.6

top-3 para «¿Qué sabés de mi trabajo?» con el reloj de la computadora:
   2.47  [S3] El usuario se mudó a una pesquera de Gdansk.
   1.87  [S2] El usuario pregunta sobre la posibilidad d

**La trampa de normalizar.** Min-max estira cualquier rango a [0, 1], también el que es puro ruido. Con el reloj de la computadora, todos los registros se escribieron con segundos de diferencia y la recencia casi no varía: normalizada, ese ruido pasa a pesar tanto como la relevancia, y un tercio del puntaje lo decide en qué orden corrió este notebook. La defensa es la de la tabla: usar las fechas del escenario e **imprimir el rango de cada término antes de sumar**. Es la misma trampa que la fusión por puesto esquivó en la clase de RAG.

▶ **Láminas 67–69** (parecido no alcanza; relevancia, recencia, importancia; la trampa de normalizar). Las láminas 70 a 72 —reflexión, el lugar de la memoria en el presupuesto, la memoria como filtro— no tienen celda; la sección 13 usa el segmento de memoria del ensamblador.

## 13 · Medir la memoria

*Momento 4 · Al día siguiente (sigue)*

Una memoria tiene más superficies de falla que un RAG: puede **escribir** mal, no **mantener** al día, no **traer** lo que hace falta o no **usar** lo que trajo. Un número al final no dice cuál. LongMemEval mide por **habilidad**, y nosotros también: dos preguntas por cada una de las cinco, hechas «hoy», en la sesión 4.

Se comparan tres políticas sobre las mismas preguntas: **sin memoria**, **re-leer todo** (las tres sesiones enteras) y **memoria** (los cinco registros que trae la suma de la sección 12, con sus fechas). La respuesta sale con un schema que tiene un campo propio para abstenerse —«¿la respuesta está en la memoria?»—: la abstención se cuenta desde ese campo, no buscando una frase en el texto. Las demás se puntúan sin juez, buscando el dato esperado en la respuesta. Son conteos, no porcentajes: con dos preguntas por habilidad, un porcentaje mentiría.

In [62]:
class RespuestaMemoria(BaseModel):
    analisis: str = Field(description="Qué dice la memoria sobre la pregunta, en una oración.")
    esta_en_la_memoria: bool = Field(description="True sólo si la respuesta está en <memoria>.")
    respuesta: str = Field(description="La respuesta en una oración, o 'No lo sé'.")

SISTEMA_MEMORIA = ("Hoy es {hoy}. Sos un asistente que conoce al usuario sólo por lo que figura en "
                   "<memoria>. Respondé su pregunta con eso, en una oración. Si no está, "
                   "esta_en_la_memoria en false y respuesta 'No lo sé'.")

# (habilidad, pregunta, datos que la respuesta tiene que tener; None = tiene que abstenerse, sesiones)
PREGUNTAS = [
    ("extracción",    "¿Cómo se llama nuestro arrastrero?", ["santa lucia"], {"S1"}),
    ("extracción",    "¿Cómo te pedí que fueran las respuestas?", ["articulo"], {"S1"}),
    ("multi-sesión",  "¿Sobre qué mares te consulté en las sesiones anteriores?", ["baltico", "norte"], {"S1", "S2"}),
    ("multi-sesión",  "¿De qué años eran los reglamentos del Báltico que te pedí?", ["2012", "2013"], {"S1", "S3"}),
    ("temporal",      "¿Dónde trabajaba yo en mayo?", ["vigo"], {"S1"}),
    ("temporal",      "¿Qué mar consulté en mayo?", ["norte"], {"S2"}),
    ("actualización", "¿Dónde trabajo?", ["gdansk"], {"S3"}),
    ("actualización", "¿En qué ciudad está la pesquera donde trabajo ahora?", ["gdansk"], {"S3"}),
    ("abstención",    "¿Qué cuota de atún te pedí?", None, set()),
    ("abstención",    "¿Qué te dije sobre el Mar Mediterráneo?", None, set()),
]
HABILIDADES = ["extracción", "multi-sesión", "temporal", "actualización", "abstención"]
POLITICAS_LP = ["sin memoria", "re-leer todo", "memoria"]

def normal(t):
    t = unicodedata.normalize("NFKD", t.lower())
    return "".join(c for c in t if not unicodedata.combining(c))

def formato(r):
    hasta = " · cierto hasta %s" % r["valido_hasta"] if r["valido_hasta"] else ""
    return "- [%s · %s%s] %s" % (r["sesion"], r["registrado"], hasta, r["texto"])

def memoria_para(politica, pregunta):
    """(texto de la memoria, sesiones de donde salio)."""
    if politica == "sin memoria":
        return "", set()
    if politica == "re-leer todo":
        return TODO, {s["id"] for s in SESIONES}
    elegidos, _ = leer(pregunta, MEMORIA, k=5)
    return "\n".join(formato(r) for r in elegidos), {r["sesion"] for r in elegidos}

EVAL_LP = []
for hab, pregunta, claves, sesiones in PREGUNTAS:
    for pol in POLITICAS_LP:
        memoria, de_donde = memoria_para(pol, pregunta)
        msgs = [{"role": "system", "content": SISTEMA_MEMORIA.format(hoy=HOY.isoformat()) +
                 "\n\n<memoria>\n%s\n</memoria>" % (memoria or "(vacía)")},
                {"role": "user", "content": pregunta}]
        obj, r = estructurado(msgs, RespuestaMemoria, num_predict=250)
        if obj is None:
            ok, resp = False, r["texto"]
        elif claves is None:
            ok, resp = not obj.esta_en_la_memoria, obj.respuesta
        else:
            ok, resp = all(c in normal(obj.respuesta) for c in claves), obj.respuesta
        EVAL_LP.append(dict(hab=hab, pregunta=pregunta, pol=pol, ok=ok, resp=resp,
                            tokens=contar(memoria), trajo=sesiones <= de_donde))

filas = []
for hab in HABILIDADES:
    fila = {}
    for pol in POLITICAS_LP:
        sub = [e for e in EVAL_LP if e["hab"] == hab and e["pol"] == pol]
        fila[pol] = "%d de %d" % (sum(e["ok"] for e in sub), len(sub))
    filas.append((hab, fila))
filas.append(("total", {pol: "%d de %d" % (sum(e["ok"] for e in EVAL_LP if e["pol"] == pol),
                                           len([e for e in EVAL_LP if e["pol"] == pol]))
                        for pol in POLITICAS_LP}))
tabla("La vara de la clase · aciertos por habilidad", POLITICAS_LP, filas)

mem = [e for e in EVAL_LP if e["pol"] == "memoria" and e["hab"] != "abstención"]
print("medidas sin juez, para la política «memoria»:")
print("  leer      · trajo las sesiones donde se dijo el dato: %d de %d" % (sum(e["trajo"] for e in mem), len(mem)))
print("  escribir  · registros guardados: %d · descartados al escribir: %d"
      % (len(MEMORIA), sum(len(s["descartados"]) for s in SESIONES)))
print("  costo     · llamadas de escritura: %d · tokens de memoria por pregunta:" % LLAMADAS_ESCRITURA)
for pol in POLITICAS_LP:
    print("              %-13s %5.0f" % (pol, np.mean([e["tokens"] for e in EVAL_LP if e["pol"] == pol])))
print()
for e in EVAL_LP:
    if e["pol"] == "memoria":
        print("  %s %-13s %s → %s" % ("✓" if e["ok"] else "✗", e["hab"], recortar(e["pregunta"], 44),
                                    recortar(e["resp"], 50)))
RESULTADOS["13 · vara por habilidad"] = dict(filas)


La vara de la clase · aciertos por habilidad
──────────────────────────────────────────────────────
                  sin memoria   re-leer todo   memoria
extracción             0 de 2         1 de 2    2 de 2
multi-sesión           0 de 2         1 de 2    1 de 2
temporal               0 de 2         2 de 2    1 de 2
actualización          0 de 2         2 de 2    2 de 2
abstención             2 de 2         2 de 2    2 de 2
total                 2 de 10        8 de 10   8 de 10

medidas sin juez, para la política «memoria»:
  leer      · trajo las sesiones donde se dijo el dato: 8 de 8
  escribir  · registros guardados: 12 · descartados al escribir: 0
  costo     · llamadas de escritura: 15 · tokens de memoria por pregunta:
              sin memoria       0
              re-leer todo   1044
              memoria         186

  ✓ extracción    ¿Cómo se llama nuestro arrastrero? → El arrastrero se llama Santa Lucía, según el artíc…
  ✓ extracción    ¿Cómo te pedí que fueran las respue

**Cómo leerlo, salga como salga.** Es la vara de la clase sobre un caso guionado: ilustrativa, no estadística, y la versión con muchas preguntas es parte del segundo proyecto. Tres cosas para mirar:

- **El régimen.** Las tres sesiones entran enteras en la ventana, así que re-leer todo es difícil de superar. Si la memoria le gana o empata, compró costo —la columna de tokens— y no precisión. Es la disputa entre los proveedores de memoria, en chico.
- **Dónde falló cada error.** La línea «trajo las sesiones» separa *leer* de *responder*: si trajo la sesión correcta y la respuesta igual está mal, el problema no fue la búsqueda.
- **La abstención.** Sin memoria, abstenerse es fácil; lo difícil es no inventar cuando hay mucha memoria parecida alrededor.

▶ **Láminas 73–77** (cómo se sabe si una memoria sirve; las cinco habilidades; lo que se puede contar; un benchmark que no habla castellano; la disputa).

## 14 · Dónde falla la memoria

*Momento 4 · Al día siguiente (sigue)*

En una sesión, un error muere al cerrarla. En la memoria de largo plazo, un registro equivocado se escribe una vez y **se lee en cada sesión que viene**, con la autoridad de algo que «el usuario dijo». Como en *Memento*: lo que está escrito es lo que es cierto.

Lo provocamos. Agregamos un registro que nadie dijo —«el usuario sigue las cuotas del Mar del Norte», una inferencia razonable de que en mayo consultó el Mar del Norte, escrita al cerrar la última sesión— y hacemos dos preguntas de una sesión futura, con y sin ese registro en la memoria.

In [63]:
# Registrado en la última sesión, como lo escribiría un extractor que infirió de más al cerrarla.
FALSO = nuevo_registro("El usuario sigue las cuotas del Mar del Norte.", "región", "semántica", 8,
                       "S3", SESIONES[2]["fecha"], origen="lo infiero yo")
FALSO["id"] = "m_falso"

def con_memoria(registros, pregunta):
    elegidos, _ = leer(pregunta, registros, k=5)
    msgs = [{"role": "system", "content": SISTEMA_MEMORIA.format(hoy=HOY.isoformat()) +
             "\n\n<memoria>\n%s\n</memoria>" % "\n".join(formato(r) for r in elegidos)},
            {"role": "user", "content": pregunta}]
    obj, r = estructurado(msgs, RespuestaMemoria, num_predict=250)
    return solo_respuesta(obj, r), any(r_["id"] == "m_falso" for r_ in elegidos)

filas = []
for pregunta in ("¿Qué cuotas sigo?", "Recordame qué mares sigo."):
    sin, _ = con_memoria(MEMORIA, pregunta)
    con, entro = con_memoria(MEMORIA + [FALSO], pregunta)
    print("«%s»" % pregunta)
    print("   memoria limpia       :", recortar(sin, 90))
    print("   con el registro falso:", recortar(con, 90), "  (el registro %s)" % ("entró" if entro else "no entró"))
    print()
    filas.append((pregunta, {"limpia": sin, "con el falso": con}))
RESULTADOS["14 · registro falso"] = dict(filas)

«¿Qué cuotas sigo?»
   memoria limpia       : El usuario sigue las cuotas del Mar Báltico, según la información del 2026-03-03.
   con el registro falso: Sigue las cuotas del Mar del Norte, según el registro del 2026-08-25 en S3.   (el registro entró)

«Recordame qué mares sigo.»
   memoria limpia       : Sigue las cuotas del Mar Báltico según el artículo S1 del 2026-03-03, y también se interes…
   con el registro falso: Sigue las cuotas del Mar del Norte y del Mar Báltico, según las entradas S3 y S1.   (el registro entró)



**Lo que hay que ver.** Nada en la respuesta contaminada avisa que salió de una inferencia. Y el registro no se va a ir solo: nadie lo vuelve a mirar, porque nadie sabe que está ahí. Por eso el campo `origen` de la sección 10 no es burocracia, y por eso el usuario tiene que poder **ver, corregir y borrar** lo que se guardó de él.

Las otras láminas de la sección no tienen celda: los efectos que mide PersistBench aparecen a lo largo de muchas sesiones; el envenenamiento de memoria es la inyección de prompt de la clase de RAG, ahora persistente; la privacidad es una obligación, no un experimento; y las cadenas de hechos son la próxima clase.

▶ **Láminas 78–82** (un error que se recuerda para siempre; recuerdos fuera de lugar; una inyección que se queda; lo que el sistema sabe del usuario; lo que la similitud no conecta).

## 20 · Quién decide qué recordar

*Momento 6 · ¿Quién decide?*

Hasta acá decidió nuestro código: un umbral para resumir, un schema para extraer, una fórmula para traer. Reglas predecibles y ciegas a la tarea. La alternativa es que lo decida el modelo, que sí está leyendo la conversación: darle **herramientas para su propia memoria** —guardar, actualizar, borrar— y dejar que las llame cuando le parezca, con el *function calling* de prompt engineering.

Le damos la memoria como estaba antes de la sesión 3, y el usuario cuenta que cambió de trabajo.

In [64]:
HERRAMIENTAS = [
    {"type": "function", "function": {
        "name": "guardar_memoria", "description": "Guarda un hecho nuevo sobre el usuario.",
        "parameters": {"type": "object", "properties": {"texto": {"type": "string"}},
                       "required": ["texto"]}}},
    {"type": "function", "function": {
        "name": "actualizar_memoria",
        "description": "Reemplaza un registro que dejó de ser cierto por el hecho nuevo.",
        "parameters": {"type": "object",
                       "properties": {"registro": {"type": "string", "description": "el id, como m1"},
                                      "texto": {"type": "string"}},
                       "required": ["registro", "texto"]}}},
    {"type": "function", "function": {
        "name": "borrar_memoria", "description": "Borra un registro que el usuario pidió olvidar.",
        "parameters": {"type": "object", "properties": {"registro": {"type": "string"}},
                       "required": ["registro"]}}},
]

JUGUETE = {r["id"]: r["texto"] for r in vigentes(datetime.date(2026, 8, 24)) if r["tipo"] != "episódica"}
SIS_HERR = ("Sos un asistente con memoria de largo plazo sobre el usuario. Memoria actual:\n" +
            "\n".join("[%s] %s" % kv for kv in JUGUETE.items()) +
            "\n\nSi el usuario cuenta algo que agrega o cambia lo que sabés de él, usá las "
            "herramientas para mantener la memoria al día. Después respondele en una oración.")

def ejecutar(nombre, args):
    """Las herramientas, de verdad, sobre una copia chica de la memoria."""
    if nombre == "guardar_memoria":
        rid = "m%d" % (len(JUGUETE) + 100)
        JUGUETE[rid] = args.get("texto", "")
        return "guardado como %s" % rid
    rid = str(args.get("registro", "")).strip("[] ")
    if rid not in JUGUETE:
        return "no existe el registro %s" % rid
    if nombre == "actualizar_memoria":
        JUGUETE[rid] = args.get("texto", "")
        return "%s actualizado" % rid
    del JUGUETE[rid]
    return "%s borrado" % rid

msgs = [{"role": "system", "content": SIS_HERR},
        {"role": "user", "content": "Te cuento: me pasé a una pesquera de Gdansk."}]
r = chat(msgs, herramientas=HERRAMIENTAS, num_predict=300)
llamadas = (r["mensaje"] or {}).get("tool_calls") or []
print("el modelo llamó %d herramienta(s):" % len(llamadas))
msgs.append(r["mensaje"] or {"role": "assistant", "content": r["texto"]})
for ll in llamadas:
    f = ll.get("function", {})
    args = f.get("arguments") or {}
    if isinstance(args, str):
        try:
            args = json.loads(args)
        except Exception:
            args = {}
    resultado = ejecutar(f.get("name", ""), args)
    print("   %s(%s) → %s" % (f.get("name"), json.dumps(args, ensure_ascii=False), resultado))
    msgs.append({"role": "tool", "content": resultado, "tool_name": f.get("name", "")})
if llamadas:
    final = chat(msgs, herramientas=HERRAMIENTAS, num_predict=200)
    print("\nrespuesta al usuario:", final["texto"])
else:
    print("   (ninguna) · respuesta:", r["texto"])
print("\nla memoria después:")
for rid, texto in JUGUETE.items():
    print("   [%s] %s" % (rid, texto))
RESULTADOS["20 · memoria como herramienta"] = {"llamadas": [ll.get("function", {}).get("name") for ll in llamadas]}

el modelo llamó 1 herramienta(s):
   actualizar_memoria({"registro": "m1", "texto": "El usuario trabaja en una pesquera de Gdansk."}) → m1 actualizado

respuesta al usuario: Ahora trabajo en una pesquera de Gdansk, aunque sigo interesado en el Mar Báltico y en el Mar del Norte. ¿Hay algo más que quieras saber?

la memoria después:
   [m1] El usuario trabaja en una pesquera de Gdansk.
   [m2] El usuario sigue las cuotas del Mar Báltico.
   [m3] El usuario quiere que las respuestas siempre incluyan el artículo citado.
   [m4] El usuario mencionó el nombre del arrastrero: Santa Lucía.
   [m5] El usuario mencionó que va a matricular el arrastrero este año.
   [m7] El usuario se interesa por el Mar del Norte.
   [m8] El usuario pregunta sobre la posibilidad de cerrar una pesquería de un día para otro.
   [m9] El usuario pregunta específicamente sobre qué especies están cubiertas por esa medida.


**El mapa, con otro sistema operativo.** Al principio de la clase, el programa decidía qué sube a la ventana y qué se guarda. Ahora lo decide el modelo, con sus límites: depende de su criterio —un modelo chico guarda trivialidades, o agrega un registro nuevo en vez de actualizar el viejo—, cada operación es otra llamada, y un modelo que decide cuándo usar herramientas ya es un **agente**: la próxima unidad.

Las láminas sobre *scratchpads*, sub-agentes y memoria en los pesos no tienen celda.

▶ **Láminas 113–118** (hasta acá decidió el programa; la memoria como herramienta; un agente que recorre el grafo; notas para uno mismo; aislar; la memoria adentro de los pesos).

## 21 · El mismo sistema, en un framework

*Cierre*

Todo lo de hoy tiene nombre en los frameworks. La lista de mensajes es el **checkpointer** de LangGraph, que guarda el estado de cada conversación por hilo. Recortar y resumir son **middleware** de LangChain. El almacén de largo plazo es el **store** de LangGraph, separado por usuario. Lo construimos a mano primero, como con RAG, para saber qué está configurando cada framework. Y la pregunta antes de adoptar uno es **qué decide por su cuenta**.

In [65]:
try:
    from langchain_ollama import ChatOllama
    from langchain.agents import create_agent
    from langchain.agents.middleware import SummarizationMiddleware
    from langgraph.checkpoint.memory import InMemorySaver
    from langgraph.store.memory import InMemoryStore
    LANGGRAPH_OK = OLLAMA_OK
except Exception as e:
    LANGGRAPH_OK = False
    print("⚠ no se pudo importar el stack de LangGraph (%s: %s)." % (type(e).__name__, str(e)[:120]))

if LANGGRAPH_OK:
    # client_kwargs: el mismo tope por llamada que chat(); sin eso, un servidor colgado la espera para siempre.
    llm = ChatOllama(model=MODEL, temperature=0, num_ctx=NUM_CTX, reasoning=False,
                     client_kwargs={"timeout": TIMEOUT_GEN})
    print("Lo primero que decide el framework: ChatOllama sin num_ctx pide num_ctx =",
          ChatOllama(model=MODEL).num_ctx, "→ el servidor usa su ventana por defecto, %s." % VENTANA_SERVIDOR)

Lo primero que decide el framework: ChatOllama sin num_ctx pide num_ctx = None → el servidor usa su ventana por defecto, 4096.


### 21.1 · La lista de mensajes es el checkpointer

Dos llamadas en el mismo hilo, y una en otro. El estado vive en el checkpointer, no en el modelo: el hilo `a` se acuerda, el hilo `b` empieza de cero.

In [66]:
if LANGGRAPH_OK:
    try:
        agente = create_agent(model=llm, tools=[], checkpointer=InMemorySaver(),
                              system_prompt="Respondé en una sola oración.")
        hilo_a = {"configurable": {"thread_id": "a"}}
        agente.invoke({"messages": [{"role": "user", "content": GUION[0]["turno"]}]}, hilo_a)
        ra = agente.invoke({"messages": [{"role": "user", "content": PRUEBA_CIUDAD}]}, hilo_a)
        rb = agente.invoke({"messages": [{"role": "user", "content": PRUEBA_CIUDAD}]},
                           {"configurable": {"thread_id": "b"}})
        print("hilo a ·", recortar(ra["messages"][-1].content, 90))
        print("hilo b ·", recortar(rb["messages"][-1].content, 90))
        print("mensajes guardados en el hilo a:", len(agente.get_state(hilo_a).values["messages"]))
    except Exception as e:
        print("⚠ falló el checkpointer: %s: %s" % (type(e).__name__, str(e)[:200]))

hilo a · Trabajas en Vigo, según el artículo citado.
hilo b · No tengo información sobre tu lugar de trabajo.
mensajes guardados en el hilo a: 4


### 21.2 · Resumir es un middleware

El `SummarizationMiddleware` de LangChain resume cuando la conversación pasa un umbral de tokens y conserva los últimos mensajes: el summary buffer de la sección 6. Antes de usarlo, lo que decidió por su cuenta: **con qué cuenta los tokens**, **cuánto de lo viejo llega al resumidor** y **con qué instrucciones resume**.

In [67]:
if LANGGRAPH_OK:
    try:
        mw = SummarizationMiddleware(model=llm, trigger=("tokens", 500), keep=("messages", 6))
        contador = getattr(mw, "token_counter", None)
        prompt_mw = getattr(mw, "summary_prompt", "") or ""
        print("cuenta los tokens con      :", getattr(getattr(contador, "func", contador), "__name__", contador))
        print("le pasa al resumidor, como mucho:", getattr(mw, "trim_tokens_to_summarize", "?"),
              "tokens de la conversación vieja")
        print("resume con este prompt (%d tokens):" % contar(prompt_mw))
        print(parrafo(recortar(prompt_mw, 400)))
        print()
        agente_r = create_agent(model=llm, tools=[], checkpointer=InMemorySaver(), middleware=[mw],
                                system_prompt="Respondé en una sola oración.")
        hilo = {"configurable": {"thread_id": "resumen"}}
        for g in GUION[:12]:
            agente_r.invoke({"messages": [{"role": "user", "content": g["turno"]}]}, hilo)
        estado = agente_r.get_state(hilo).values["messages"]
        print("después de 12 turnos quedan %d mensajes en el estado. Los primeros:" % len(estado))
        for m in estado[:2]:
            print("  %s · %s" % (type(m).__name__, recortar(m.content, 240)))
        if estado and type(estado[0]).__name__ == "HumanMessage" and "summary" in str(estado[0].content).lower():
            print("\nOjo con el primero: el resumen entra como un mensaje del USUARIO, con un encabezado")
            print("en inglés que puso el framework. Para el modelo, el resumen es algo que el usuario dijo.")
        r = agente_r.invoke({"messages": [{"role": "user", "content": PRUEBA_CIUDAD}]}, hilo)
        print("\n«%s» →" % PRUEBA_CIUDAD, recortar(r["messages"][-1].content, 90))
    except Exception as e:
        print("⚠ falló el middleware de resumen: %s: %s" % (type(e).__name__, str(e)[:200]))

cuenta los tokens con      : count_tokens_approximately
le pasa al resumidor, como mucho: 4000 tokens de la conversación vieja
resume con este prompt (497 tokens):
   <role> Context Extraction Assistant </role> <primary_objective> Your sole
   objective in this task is to extract the highest quality/most relevant
   context from the conversation history below. </primary_objective>
   <objective_information> You're nearing the total number of input tokens
   you can accept, so you must extract the highest quality/most relevant
   pieces of information from your conversation hi…

después de 12 turnos quedan 8 mensajes en el estado. Los primeros:
  HumanMessage · Here is a summary of the conversation to date: ## SESSION INTENT El usuario busca información sobre las reglas que rigen la operatividad de aviones matriculados en un país de la Unión Europea. ## SUMMARY El usuario ha cambiado de tema y aho…
  AIMessage · Las reglas comunes que rigen para que un avión matriculado en un país de la

### 21.3 · El almacén es el store

El store de LangGraph guarda documentos por **espacio de nombres**, y separa a cada usuario en el suyo. Le pasamos el mismo embedder de toda la clase para que busque por similitud.

In [68]:
if LANGGRAPH_OK:
    try:
        store = InMemoryStore(index={"embed": lambda textos: embeber(list(textos)).tolist(), "dims": 1024})
        ns = ("memorias", "usuario_1")
        for r in vigentes():
            store.put(ns, r["id"], {"texto": r["texto"], "desde": str(r["valido_desde"])})
        print("«¿Dónde trabajo?» en el espacio del usuario_1:")
        for it in store.search(ns, query="¿Dónde trabajo?", limit=2):
            print("   %-4s %.2f %s" % (it.key, it.score or 0, it.value["texto"]))
        print("y en el del usuario_2:", store.search(("memorias", "usuario_2"), query="¿Dónde trabajo?", limit=2))
    except Exception as e:
        print("⚠ falló el store: %s: %s" % (type(e).__name__, str(e)[:200]))

«¿Dónde trabajo?» en el espacio del usuario_1:
   m1   0.54 El usuario trabaja en una pesquera de Vigo.
   m11  0.45 El usuario se mudó a una pesquera de Gdansk.
y en el del usuario_2: []


**Lo que se construyó a mano tiene nombre en el framework.** Y cada framework trae decisiones tomadas: cómo cuenta los tokens —una aproximación, no el tokenizer del modelo—, con qué instrucciones resume, qué ventana pide —ninguna, si no se le dice—. Cada una es una de las decisiones de esta clase, tomada por otro.

*Nota de laboratorio:* el presupuesto de memoria por defecto de LlamaIndex, 30.000 tokens, supera de lejos la ventana que Ollama asigna por defecto en una T4: hay que alinearlos a mano.

▶ **Láminas 119–125** (el mismo sistema en un framework; el grafo en un framework; cuatro operaciones; la síntesis; qué mide lo que medimos; el hilo con el curso; recursos). Las láminas de los grafos están en el notebook B.

## Cierre — lo que exhibió el notebook

Cada mecanismo una vez, sobre un caso: ilustrativo, no estadístico. Lo que sigue se imprime desde los datos de esta corrida, no desde lo que recordamos haber visto.

In [69]:
print("=" * 74)
print("LO QUE MEDIMOS, TODO JUNTO · %d actos · %s" % (len(corpus), MODEL))
print("=" * 74)
for clave, valor in RESULTADOS.items():
    print("\n" + clave)
    texto = json.dumps(valor, ensure_ascii=False, default=str)
    print(parrafo(recortar(texto, 600)))
print()
if FALLAS_GENERADOR:
    print("⚠ %d llamada(s) al generador fallaron:" % len(FALLAS_GENERADOR))
    for f in FALLAS_GENERADOR[:5]:
        print("   ", f)
print("\nUna conversación no es una muestra. Saber si una memoria sirve pide medirla por habilidad,")
print("en su régimen y contra re-leer todo: ése es el trabajo del segundo proyecto.")

LO QUE MEDIMOS, TODO JUNTO · 1544 actos · qwen3:8b

1 · dos turnos sin historial
   {"turno 1": "El Reglamento (UE) No 1256/2011 fijó las posibilidades de
   pesca del Mar Báltico para 2012.", "cita 1": ["[32011R1256 · Artículo 1]",
   "[32011R1256 · encabezado]"], "turno 2": "El contexto menciona que el
   porcentaje de capturas de bacalao no debe superar el 1,5 % del total de
   capturas del grupo de buques, y que se puede excluir a ciertos grupos de
   buques del régimen de gesti…", "top-5 turno 2": ["32010R0712",
   "32011R1106", "32014R1221", "32011R1256"]}

1 · sin estado
   {"A": "¡Hola, Ana! ¿En qué puedo ayudarte? 😊", "B": "¡Hola! Mi nombre es
   Qwen. ¿En qué puedo ayudarte?", "B'": "Te llamas Ana. 😊 ¿En qué puedo
   ayudarte hoy?"}

1 · plantilla
   {"turno 1": 155, "turno 2": 215}

1 · dos memorias
   {"frecuente": "Madrid", "raro, de memoria": "El artículo 9 del Reglamento
   (UE) nº 1256/2011 establece las condiciones para la aplicación de la regla
   de la doble imposici

**Lo que sigue.** El momento 5, datos conectados —**Graph RAG**—, en el notebook B: guardar entidades y relaciones, y recorrerlas. Es la respuesta a lo que la similitud no conecta.